# Домашняя 2 · Альянс: слить два ранжирования и не обмануть себя

**Неделя 9 · занятие 1.** Опора — L12 «Альянс»: позднее взаимодействие (ColBERT), SPLADE,
гибрид, Learning to Rank.

На неделе 7 мы построили каскад и получили честный ответ «не отличимо от нуля». Сегодня
добавляем третью силу — слияние лексического и плотного — и впервые за курс получим результат,
который **отличим**. Заодно увидим ровно то место, где такие результаты подделываются
без всякого злого умысла.

| # | вопрос занятия | чем отвечаем |
|---|---|---|
| 1 | Есть ли что брать от объединения двух ретриверов? | считаем потолок объединения — до всякого слияния |
| 2 | RRF или слияние скоров? | считаем оба на своих данных и сравниваем с потолком |
| 3 | Как подобрать вес и не подогнать под шум? | делим запросы на подбор и проверку, меряем оптимизм |

**Данные.** Артефакты недели 7: выдачи и **скоры** BM25, би-энкодера и кросс-энкодера на 1500
документах 20NG с псевдозапросами. Плюс игрушки из `data/l8-*.json` для сверки с доской.
Если артефактов нет, ноутбук соберёт уменьшенную версию сам и скажет об этом.

**Среда.** Colab T4 через VS Code. Слияние — чистая арифметика на CPU, секунды. Модели нужны
только в запасном пути, если артефактов недели 7 не нашлось.

**Бюджет: ≈120 минут.**

**Артефакт на вынос.** `artifacts/fusion.json` — слитые выдачи, подобранный вес и честная оценка
на отложенных запросах. На неделе 10 ANN-индекс будет строиться под то ранжирование, которое
мы сегодня выберем.

**Как запускать.** Сверху вниз.

<details><summary>Почему вообще складывают два ретривера, а не выбирают лучший</summary>

Потому что у них **разные типы промахов**, и это единственная причина, по которой сложение
вообще может помочь. Сложение двух систем с одинаковыми ошибками не даёт ничего.

**Чем промахивается лексический поиск.** Он не находит документ, в котором нет слов запроса.
Синонимия, морфология, перефразировка — всё мимо. Зато он **никогда** не промахивается
по точному совпадению: артикул, номер ошибки, редкая фамилия найдутся всегда.

**Чем промахивается плотный.** Он размывает редкое и точное: артикул `X7-4412B` для энкодера
почти неотличим от `X7-4413B`, потому что оба — редкие токены без семантики. Зато он находит
документ про космос по запросу про орбитальные аппараты.

**Отсюда следствие, которое стоит запомнить.** Гибрид почти всегда не хуже лучшего из двух —
не потому, что «два лучше одного», а потому, что множества найденного пересекаются лишь
частично. Если бы они совпадали, слияние было бы бесполезно, и это можно **проверить заранее**,
одним замером, до всякой реализации. Именно с него мы и начнём.

**Чего гибрид не делает.** Он не чинит общий промах. Документ, которого не нашла ни одна
из двух систем, не появится ни от какого слияния — и этот потолок мы тоже посчитаем.
</details>

## Шаг 0 · Пины и preflight

In [ ]:
# ПИНЫ — точнее, ОТКАЗ от них там, где они ломают Colab.
# Базовый стек образа (numpy, scipy, scikit-learn, matplotlib, torch) собран сам под себя.
# Понижать его нельзя: `pip install numpy==1.26.4` откатывает ОДИН numpy, а scipy и sklearn
# остаются собранными под numpy 2 — и первый же импорт падает с
# «ModuleNotFoundError: No module named 'numpy.char'». Ставим ТОЛЬКО то, чего в образе нет.
import importlib.util as _ilu, subprocess as _sp, sys as _sys

NEEDED = {"sentence_transformers": "sentence-transformers"}
_missing = [pkg for mod, pkg in NEEDED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print("ставлю:", ", ".join(_missing))
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    print("готово · если следующий импорт упадёт — Runtime → Restart session, потом эта ячейка снова")
else:
    print("всё нужное уже в образе Colab — ставить нечего")

import json, math, os, random, re, statistics
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ДАННЫЕ ЛЕКЦИИ. Занятие сверяет свои результаты с числами лекции, а те живут в папке
# data/ курса. В Colab её нет — и раньше ноутбук падал на первой же сверке с
# FileNotFoundError. Файлы крошечные, поэтому вшиты прямо сюда: ниже точная копия нужных
# data/*.json, сжатая zlib и записанная base64. Гейт _research/check_notebooks.py следит,
# чтобы копия совпадала с оригиналом байт в байт, — так что разойтись они не могут.
# Если рядом уже лежит настоящая папка курса (или задан DLS_DATA), берётся ОНА.
import base64 as _b64, os as _os, zlib as _zlib
from pathlib import Path as _Path

_DATA_DIR = _Path(_os.environ.get("DLS_DATA", "./data"))
_EMBEDDED = {
    "l8-colbert.json": (
        "eNqVVktu2zAQ3ecUA21io7L8a1rHQbtoGxQF+gHc7AJDpaWJxYQiFZJK4xbZ9gDd9EC9SU/Sob6RChfJynoj6nH4+GbG"
        "3w8AvDBWkbcE7z2zCFxa1CyyXEkYw2slXp2uzuADu/3M0wDO1A5eQMJkPIpUmuWWbQSCVVcof/8qfiBShkuElFnNIzQw"
        "MDYWfDNcViT0vcnTkNOK2/DSLR+chtfnfO3DaRifX66HAVBCKxQwC+ZH8NKhd1rDNJhPYGATBIG3PGICrGbZEhhkqC8w"
        "snC4YfLq0G0dJcCEojS4AaksoFT5NiHmFdJ3L+BCq2+Ua3W8kbG7+hhgeGp8MJmg7GPY7CDUaJDpKBlvUYZiQZiJINsF"
        "nl+oZ1SuI3QC9lfSIkpY7Yiu1ACeQJcDBrXCUYLRVaZIfh/GudHjDZfjbGcTJefDcidiol2+0yOB6zOXraHAeRGgkOY3"
        "qIulBXRitOhCKBV7BVqXQa9UuaF0O+CtdQdxIhdshscImWBcQkGAccsY/5MBFF82K+qUHMn9YMF3P1BTV5F1swXdRYe+"
        "fQKYBFO/A4+78GkXzhu09vfwTY66n/Tg0x6ePYBx0UuiT9HDx0ct5z9iaPWVKqijR+fQnYTvcbUMaVGAxOAKqwjetVag"
        "EttnBeckYFmm1Q2VBAOhmHycDzpeJNxhIFyTP9YC/Sub+f9zyGTxeBMc96+wt8PR4ymnD8/yYSaYde59cR9NZ/tdMK2K"
        "4u6gMoLnulLbYlIqSge9SIkNajvielw93syCiVd7pySbVft6dJOX1I3pMpfUoXOswlLplAn+rR/PGO8Y7zpH7fpc2TtK"
        "75XNq3Vc3bge2Khqc+83c6cYSoHKHebB5FldtdWLkmsaPF9Mjpuz2WIAOP2amUlTaVSOFEw3GMdcbs0SileCGRsmPI5p"
        "GBjrxm4jGjx/tvjz4yfJ6WaPS7idDIcGvggarkwHX5FvE/vFdyskvJ+NWnlP9g7aE8AbJgbDJ1KFW81iekqZzJkIDWI8"
        "mAxPaFWacuvyKEdkOfJp3kNxNrbViEB/DT6t3pyu3n18C4NGLZrUjUBDvxi7KdtKbvMYA6fw3cHdwV9e0Bjx"
    ),
    "l8-hybrid.json": (
        "eNqNlMGO2jAQhu/7FKP0kmghEEJWaKs9lIaqlbpSBXurKuTYZnFjbGQ7bBFC6kP0CfsknSSQpiVFvUTjsf39M5PxHG4A"
        "vCXT1LsH7/0+M4KB4c4IviMSBjCfvwvhSe/hAdZEsT7Vm23hSCY5HqNiazQlsm+IymFVWKFVD/KHuyH41jEpsuAeLNWG"
        "4/Vo4Oe3ZrElxvIAbk/rlCtcosSaA9VoK1tYwHggHYE/DIfxKAngRSj7GhyesdV9WJGdLoxwHNIIfn7/AazkQBmHBeFA"
        "Eusq/4pIacFpiA2redEkCL1elbZFBuVl5kvDLSeGrgfPXC3lJNzuwXd634NTHiG8RVJGaA7T2Yc5qGKTcWNBih0HoYAR"
        "RwYy7mdc0fWGmNyGX61W4CvtgBVbKShxnJ2lcxS9G1Ym5mpx9RltXKVRdaC0Ro0VN9a4sRIPjS8Voa4JMg71niQZl2VW"
        "08dRgv9w8enjm3R2vqkN46YRbEu2RduybeFGGsXxe6xTKIvfob+YzuZP4Ffbwb/1/0+z14r3L33sPM4a5OF8UJS+P/Cn"
        "9kP36LcvPUUfNa6qZdFTt1/lPfY62XEXO75kj7rZ19BRFzq6RCed6GhyjT3uYo8v2XE3+2pJki52csked7NrdNPb+Hx4"
        "XQ4Qtj0BXkXgS/4Nn5WEDXF0HUBWuK45UA+Oi+FSzpSwnG84yF6IYRbIs+F8w5ULvZvjzS+gE0L0"
    ),
    "l8-ltr.json": (
        "eNptUktu2zAQ3fsUA28qIzIt+VengRZFAhQt3DRIswsCgRYpia4kGiSNwAiyClB0HfQuXXTXAyR3yEk6pGRbbrPjvMd5"
        "83t3HYBuzGTSfQfdOaeqElUGRsIlrb4RuJIbiCCnFesnslytDV0U3HHn3IAWWSkFgyOY03LBqMXh6Wd1dvoBbrnIcmPF"
        "PG1YIRY9ssu7iJ4fPB2Llx+POl72bBSSYS8KyNvpbOJDIrXBYDgdjXzIFGWCVyYK8fvzg8VH4eQETM6hFLovFeOKM7he"
        "+uIGW9Xg6gdkOgqOffdN39IVFCI1GoSxw4UkgJfvj9teAzKaHocn8PQbh93W+/OrYSPkg9lkjNvIFadM7+awEq3J9+Hn"
        "95dXpOu73Wq5Vgm3640V17jhJB9kvIqLGVltwDNy48NuQ64gEzqR68pAOChkNvQUih+FPRAa5uM3GlKpynVBwWPU0EEx"
        "7pfcKJFostSy6jV1URdr3uETgxUVahdhjPf+2IoREcx2KFxuAyleIBa2EGxL2UlCMmnAe78l+ek1yeX/ksErkniCrWSn"
        "JVzzZyJNXdlhg6r6ABdKLlyus80BdYoWcpQzUU1tL1vjaKIGr1iStbeTrJXCb1+ssRC/3nd7MIzoNu8b/59MV8DZb0vQ"
        "1HD1FW3oxgj2W+OFoed1/dqFB9MXzk2Osw7s2O3cd/4CaRAFBA=="
    ),
    "l8-splade.json": (
        "eNqtVs1u4zYQvucpBrrERixZlixbbpDDAt1DgewPdlOgQLEwaImxhFCkQlJx3CDXPkAfsU/SIaXYkmNttvBebI1Ift/M"
        "fMPRPJ0BOMtUJM4v4FxTIjlNQZVEKgpj+Pr5+t2v7z24EVu4gozw1E1EUVaarBiFDc3XmVawwTUm1oPJxRd6/fsAH3M9"
        "HIJ4oBIIzNyNkCk8iISsKkbkFi5AZ/SFJBUaBkqnLF8NkQgX7iuKm85lbs7fMiHSc1hLsVHw/o/P7z5+/e3TR9BUFgpW"
        "hN+NNwQNGJRC5RqPNF6NgCMwy3GNMAQ2NFcQev50MYVcgRQVBoN+cFdVBQwoSTJIc1UyssUMlFKkVaLrbWhrAVNIy5Hx"
        "nAOeKGh6af5rCLsNNqJiKayNE5YoRN4vlDAkvpXiLzxY5xPRyjqCEWaB5QkSrLawlFShAEk2XlO+ZDHahHnl1nNGViQl"
        "KplQo9PhTtwEAy22CFcnElPcxYBBQ51kNLkrRc4xQ+NKyfEq5+NyqzPBw2FNhEBI8oSPaFjZ0PzTmvjCymI3WtNIsLes"
        "WnvTSrM3mSB8bxWC061jrW/1S8cqv+M2rtBHbQJu1UIbDutMtXwDCDx/tDN8Lxq1VlrGxAv2hjtpH3IRojG+7ZgkZdXJ"
        "PH73zGuW5jp1iNC3RTxrn5v6UZchCuL2+jyOo/9BSh9LwlUueIe2K+tOyZfT9v+5Ea3uHa8kM1fcymagbC9A+TZY6Ztc"
        "Z3BQGcdC971FJ662MWtnIHw73I7D9ua1qJ72Ye6LrR38Pb49FMJJ8WXHRcf0jPplHM+b18+jHpLDBN/bk11xG475MY4g"
        "DhdvcXTvyy6QbsU0JLNjJPNo8mYgXSV3kXTL0JJ0lHohwbwG0eSFpNMNsGXjhrplnzUeOKaf7btTIVJqTIcT1GxsundK"
        "8ROVCOzaXFHpmt8CP1bY2nXOnINGc6Sx1Gs3orz5Tpng0tFKqesYlwIvns/b4bbv2S1hin4vrw38K/3a8OFsfiK89V71"
        "4Ee+H/Tga1n9sPc5X/cQBEE4/wkEx/3HUovC8DR4pQU+HEPH6p5Mpqehmy+4fBC57GGI4sWJ6VkRiTXegx5Fi9PQqRT2"
        "QA++vzgx+6/aSgt9GkeT09AZuaN94LPZ/Ce4jpeQ9TEE4Y9crU4vxGHWTn4frj+4GSUp1MMP/Pv3P63ReziCgjy6pRDY"
        "8Or5ux6LBVeXZup078ygWftyCfSBsMHwgovlWpIUnwrCK8KWitJ04A8vAWf9Itcaser51YPznbfnV8ZZuBUSJ23uNpN2"
        "PdZ6ZlJ4Pns++w8mUv1z"
    ),
}

_DATA_DIR.mkdir(parents=True, exist_ok=True)
_new = [n for n, b in _EMBEDDED.items() if not (_DATA_DIR / n).exists()]
for _n in _new:
    (_DATA_DIR / _n).write_bytes(_zlib.decompress(_b64.b64decode(_EMBEDDED[_n])))
print(f"данные лекции: {len(_EMBEDDED)} файл(ов) в {_DATA_DIR} · "
      f"распаковано {len(_new)}, остальные уже лежали на месте")


In [ ]:
def preflight():
    problems = []
    for name in ("l8-hybrid", "l8-colbert", "l8-splade", "l8-ltr"):
        if not Path(f"{DATA_DIR}/{name}.json").exists():
            problems.append(f"нет {DATA_DIR}/{name}.json -- сверка с лекцией невозможна.")
    if not CASCADE_PATH.exists():
        problems.append(
            f"нет {CASCADE_PATH} (артефакт недели 7). Это НЕ ошибка: ноутбук соберёт "
            "уменьшенную версию сам -- 400 документов вместо 1500 и 40 запросов вместо 80. "
            "Числа будут другими, и сравнивать их с числами разбора нельзя.")
    for p in problems:
        print("!", p)
    print("preflight:", "ЧИСТО" if not problems else f"{len(problems)} замечани(я/й) -- читай выше")
    return not problems

## Шаг 1 · Конфигурация

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SMOKE = os.environ.get("SMOKE", "0") == "1"
K = 100                      # глубина, на которой сливаем и меряем полноту
RRF_K = 60                   # константа RRF из статьи Cormack et al.
ALPHAS = [round(a / 10, 1) for a in range(11)]     # вес лексической ступени
DATA_DIR = os.environ.get("DLS_DATA", "./data")
ARTIFACTS = Path(os.environ.get("ARTIFACTS", "./artifacts"))
CASCADE_PATH = ARTIFACTS / "cascade.json"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

RUN = {"seed": SEED, "smoke": SMOKE, "k": K, "rrf_k": RRF_K}
print(json.dumps(RUN, ensure_ascii=False))
preflight()

**Что видно.** Конфигурация напечатана, и `preflight()` сказал, нашлись ли артефакты недели 7.
Сравнивать надо не строки вывода, а **факт наличия файла с тем, что из этого следует**: без него
ноутбук пойдёт запасным путём и посчитает всё на вчетверо меньшем корпусе, а меньший корпус —
более лёгкая задача, и все числа окажутся выше. Механизм такой по правилу накопительной системы:
переиспользуем, если есть, строим сами, если нет, и **говорим вслух**, каким путём пошли.
Чего этот вывод НЕ показывает: корректности артефактов недели 7 — мы им доверяем, потому что
там они проверялись. Что делать: если пошёл запасным путём, не сравнивай свои числа с числами
из разбора ниже.

## Шаг 2 · Загружаем выдачи и скоры

Нам нужны **скоры**, а не только порядок. Из порядка нельзя узнать, обошёл документ соседа
на волос или на пропасть, а слияние складывает именно величины.

In [ ]:
def build_fallback(n_docs=400, n_queries=40):
    from sklearn.datasets import fetch_20newsgroups
    from sentence_transformers import SentenceTransformer
    cats = ["sci.space", "rec.sport.hockey", "comp.graphics",
            "talk.politics.mideast", "sci.med", "rec.autos"]
    raw = fetch_20newsgroups(subset="train", categories=cats,
                             remove=("headers", "footers", "quotes"), random_state=SEED)
    tok = re.compile(r"[a-z]{2,}")
    sent = re.compile(r"(?<=[.!?])\s+")
    pool = []
    for t in raw.data:
        t = " ".join(t.split())
        if len(tok.findall(t.lower())) < 80:
            continue
        ss = [s for s in sent.split(t) if 12 <= len(s.split()) <= 25]
        if ss:
            pool.append((t, ss))
    random.Random(SEED).shuffle(pool)
    docs = [p[0] for p in pool[:n_docs]]
    queries = []
    for i in range(n_queries):
        text, ss = pool[i]
        s = max(ss, key=len)
        queries.append(s.strip())
        docs[i] = text.replace(s, "").strip()

    post, dl = defaultdict(dict), {}
    for i, d in enumerate(docs):
        tk = tok.findall(d.lower())
        dl[i] = len(tk)
        for t, tf in Counter(tk).items():
            post[t][i] = tf
    n, avgdl = len(docs), sum(dl.values()) / len(docs)

    def bm(q):
        sc = defaultdict(float)
        for t in tok.findall(q.lower()):
            p = post.get(t)
            if not p:
                continue
            idf = math.log((n - len(p) + .5) / (len(p) + .5) + 1)
            for d, tf in p.items():
                sc[d] += idf * (tf * 2.5) / (tf + 1.5 * (.25 + .75 * dl[d] / avgdl))
        o = sorted(sc, key=lambda d: (-sc[d], d))
        return o[:K], [sc[d] for d in o[:K]]

    bmo, bms = zip(*(bm(q) for q in queries))
    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    emb = st.encode(docs, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    qemb = st.encode(queries, normalize_embeddings=True, show_progress_bar=False)
    sims = qemb @ emb.T
    bio = [[int(d) for d in np.argsort(-sims[i])[:K]] for i in range(n_queries)]
    bis = [[float(sims[i][d]) for d in bio[i]] for i in range(n_queries)]
    return {"queries": queries, "gold": list(range(n_queries)),
            "bm25_top100": list(bmo), "bm25_scores100": list(bms),
            "bi_top200": bio, "bi_scores200": bis, "rerank25": bio, "_fallback": True}

if CASCADE_PATH.exists():
    C = json.load(open(CASCADE_PATH, encoding="utf-8"))
    source = "артефакт недели 7"
else:
    C = build_fallback(200 if SMOKE else 400, 20 if SMOKE else 40)
    source = "ЗАПАСНОЙ путь: пересчитано здесь, уменьшенный масштаб"

QUERIES, GOLD = C["queries"], C["gold"]
BM, BMS = C["bm25_top100"], C["bm25_scores100"]
BI, BIS = C["bi_top200"], C["bi_scores200"]
CASCADE = C["rerank25"]
NQ = len(QUERIES)

print(f"источник: {source}")
print(f"запросов: {NQ} · глубина выдачи BM25 {len(BM[0])}, би-энкодера {len(BI[0])}, "
      f"каскада {len(CASCADE[0])}")
print(f"скоры BM25: от {min(BMS[0]):.2f} до {max(BMS[0]):.2f} · "
      f"би-энкодера: от {min(BIS[0]):.3f} до {max(BIS[0]):.3f}")
RUN["source"], RUN["n_queries"] = source, NQ

**Что видно.** Три выдачи разной длины и две шкалы скоров, не имеющие между собой ничего общего:
BM25 выдаёт суммы по терминам без верхней границы, би-энкодер — косинусы в отрезке от минус
единицы до единицы. Сравнивать надо не диапазоны сами по себе, а **их несопоставимость**:
сложить эти числа напрямую нельзя, разница в порядке величины съест вклад одной из систем
целиком. Механизм прямой — у метрик нет общей единицы измерения. Чего этот вывод НЕ показывает:
как именно приводить их к общей шкале; вариантов несколько, и они дают **разный** результат,
что мы и увидим в части 3. Что делать: заметить, что длина выдачи каскада вчетверо меньше
остальных, и не сравнивать полноту на глубине 100 у систем, которые возвращают 25 документов.

<details><summary>Почему кросс-энкодер не участвует в слиянии, хотя он тут же лежит</summary>

Артефакт недели 7 содержит выдачу кросс-энкодера, и соблазн включить её третьим голосом
очевиден. Мы этого не делаем, и причина не в лени.

**Первое: он не ретривер.** Кросс-энкодер видел только двадцать пять кандидатов, отобранных
би-энкодером. Его выдача — подмножество плотной, а не независимый взгляд на корпус. Сливать
его с би-энкодером значит складывать систему саму с собой: уникального вклада у неё ноль
по построению, и потолок объединения это немедленно покажет.

**Второе: разные глубины ломают RRF.** У BM25 и би-энкодера по сто позиций, у кросс-энкодера
двадцать пять. Документ на двадцать шестом месте плотной выдачи получает от кросс-энкодера
не «низкий ранг», а **отсутствие вклада** — то есть ровно то же, что документ, признанный
плохим. Это молча искажает голосование в пользу верхушки короткого списка.

**Третье: он уже применён.** Кросс-энкодер — последняя ступень, и его место после слияния,
а не внутри. Правильная архитектура: слить лексическое и плотное, взять топ-`k` слияния,
переранжировать кросс-энкодером. Ровно это и соберётся на неделе 14 в финальном проекте.

**Что стоило бы проверить, будь время.** Каскад поверх слияния против каскада поверх одного
би-энкодера: помогает ли более полный вход второй ступени. Судя по неделе 7, где углубление
не помогало, — вероятно, нет, и это был бы полезный отрицательный результат.
</details>

<details><summary>Почему скоры BM25 не сопоставимы даже между собственными запросами</summary>

Мы сказали, что шкалы двух систем несопоставимы. У BM25 хуже: его скоры несопоставимы
и **внутри одной системы**, между разными запросами.

**Механизм.** Скор BM25 — это сумма по терминам запроса. Запрос из двух слов даёт вдвое меньше
слагаемых, чем запрос из четырёх, и потолок скора линейно зависит от длины запроса. Плюс каждое
слагаемое содержит `idf`, который тем больше, чем реже термин. Запрос из двух редких слов даст
скор выше, чем запрос из четырёх частых, — и это ничего не говорит о том, какой из ответов лучше.

**Что отсюда следует.** Порог вида «показывать документы со скором выше 12» не работает:
на одном запросе двенадцать — отличный результат, на другом такого скора не достигает никто.
Именно поэтому в проде пороги ставят на **нормированный** скор или на позицию, а не на сырой.

**Почему это важно именно для слияния.** Нормировку мы делаем **внутри каждого запроса** —
посмотри на код `fuse`: `zscore` вызывается на скорах одного запроса, а не всего прогона.
Это не случайность. Нормировать по всему прогону значило бы смешать запросы с разной длиной
и разной редкостью терминов, и вес `alpha`, подобранный на таком наборе, оказался бы подогнан
под распределение длин запросов, а не под соотношение сил двух систем.

**Калибровка.** Существует правильный способ сделать скоры сопоставимыми между запросами —
калибровка: обучить монотонное преобразование скора в вероятность релевантности на размеченных
данных (изотоническая регрессия или Платт). После неё скор означает одно и то же везде, пороги
работают, а слияние можно делать просто сложением вероятностей. Плата — нужна разметка,
и калибровку надо переобучать при каждом изменении системы.
</details>

⚠️ Ловушка E · **Полноту нельзя сравнивать у выдач разной длины.** Каскад недели 7 возвращает
ровно 25 документов, поэтому его `Recall@100` физически равен `Recall@25`. Поставить его
в одну таблицу с RRF@100 — значит сравнить систему с урезанной выдачей против системы с полной
и объявить победителя. Ниже мы такого не делаем, и в собственных отчётах не делай тоже:
глубина выдачи — часть конфигурации, а не свойство метода.

---

## Часть 1 · Дубликаты и потолок объединения — 20 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 1.1 | Что ломается при наивном объединении? | сливаем без дедупликации и смотрим на Recall |
| 1.2 | Есть ли что брать от второй системы? | **замер без модели**: потолок объединения |

### Шаг 1.1 · Долг недели 4, который наступил

На неделе 4 мы написали `metrics.py` и честно записали в ограничения: дубликаты не проверены,
их нет в той конструкции, но они появятся при слиянии. Они появились.

In [ ]:
def rr_at(order, gold, k=10):
    for i, d in enumerate(order[:k], 1):
        if d == gold:
            return 1 / i
    return 0.0

def recall_at(order, gold, k):
    return 1.0 if gold in order[:k] else 0.0

def naive_concat(a, b):
    return list(a) + list(b)          # как сливают, когда не думают

def dedupe(order):
    seen, out = set(), []
    for d in order:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out

naive = [naive_concat(BM[i][:K], BI[i][:K]) for i in range(NQ)]
clean = [dedupe(x) for x in naive]
dups = [len(n) - len(c) for n, c in zip(naive, clean)]

print(f"наивная склейка: длина {len(naive[0])}, из них дубликатов "
      f"{dups[0]} (в среднем {statistics.mean(dups):.1f} на запрос)")
print(f"после дедупликации: {len(clean[0])} уникальных на первом запросе")
print()
print("а теперь то, ради чего это здесь:")
counted_twice = sum(1 for i in range(NQ) if naive[i].count(GOLD[i]) > 1)
print(f"  правильный документ встречается ДВАЖДЫ в {counted_twice} запросах из {NQ}")
print(f"  Recall@200 по наивной склейке (с дубликатами): "
      f"{statistics.mean(recall_at(naive[i], GOLD[i], 2 * K) for i in range(NQ)):.4f}")
print(f"  а если бы метрика СУММИРОВАЛА попадания, как это делает наивный код:")
naive_sum = statistics.mean(min(naive[i].count(GOLD[i]), 2) for i in range(NQ))
print(f"  «Recall» = {naive_sum:.4f} -- то есть БОЛЬШЕ единицы, что бессмысленно")
RUN["dup_mean"] = statistics.mean(dups)

**Что видно.** Больше чем в половине запросов правильный документ попадает в склейку дважды,
и последняя строка показывает, чем это кончается: метрика, суммирующая попадания, выдаёт
значение больше единицы. Сравнивать надо не длины списков, а **число уникальных с числом
позиций**: в наивной склейке двести позиций и заметно меньше документов. Механизм тривиален
и потому опасен — обе системы находят одни и те же хорошие документы, это признак того, что
они согласны, а вовсе не ошибка. Чего этот замер НЕ показывает: как именно объединять; порядок
внутри склейки мы пока не задавали вовсе. Что делать: дедуплицировать **до** любой метрики
и до любого слияния. Одна функция в шесть строк, и целый класс невозможных чисел исчезает.

<details><summary>Три способа схлопнуть дубликат — и почему выбор не безобиден</summary>

Дедупликация выглядит тривиально: оставить первое вхождение. На самом деле это один
из трёх вариантов, и они дают разные ранжирования.

**Оставить лучшую позицию.** Именно это делает наш `dedupe`: документ получает свою лучшую
позицию из двух списков. Просто и агрессивно оптимистично — документ, найденный первым
у слабой системы и последним у сильной, поднимется наверх.

**Сложить вклады.** Так делает RRF: документ, встретившийся в обоих списках, получает сумму
`1/(k+r₁) + 1/(k+r₂)`. Это вознаграждает согласие и есть главная идея метода. Дубликат здесь
не проблема, а сигнал.

**Взять худшую позицию.** Консервативный вариант: документ должен быть хорош по мнению обеих
систем. Резко снижает полноту и почти не используется, но в задачах, где ложное срабатывание
дорого (модерация, юридический поиск), встречается.

**Что выбрать.** Зависит от того, что означает несогласие систем. Если системы одинаково
компетентны, но смотрят на разные аспекты — складывай. Если одна заведомо сильнее — бери
её позицию, а вторую используй только для документов, которых у первой нет. Если цена ошибки
высока — бери худшую.

**Ошибка, которую делают чаще всего.** Дедуплицируют **после** обрезки по `k`. Тогда в топ-100
попадает список из ста позиций, где часть — дубликаты, и уникальных документов оказывается,
скажем, семьдесят. Метрика при этом считается по ста позициям, и полнота занижена на треть
без всякой причины. Дедупликация всегда идёт до обрезки.
</details>

⚠️ Ловушка A · **Дубликат — не ошибка данных, а согласие систем.** Соблазн — считать
повторяющийся документ мусором и выбросить обе копии. Он несёт информацию: документ, найденный
обеими системами, скорее релевантен, и RRF ниже пользуется именно этим. Правильно не выбросить,
а **схлопнуть, сложив вклады**.

⚠️ Ловушка D · **Метрика не обязана проверять свои входы.** Наш `recall_at` вернул бы больше
единицы, если бы суммировал попадания вместо проверки вхождения. Реализация, написанная под
выдачу без дубликатов, молча даёт бессмыслицу на выдаче с дубликатами — и не падает.

### Шаг 1.2 · Замер без модели: потолок объединения

**Замер без модели.** Прежде чем реализовывать хоть какое-то слияние, ответим на вопрос,
есть ли там вообще что брать. Если объединение кандидатов двух систем не богаче лучшей из них,
никакое слияние не поможет — и это видно **до** любой реализации.

<details><summary>Шесть типов ловушек этого занятия — и новый, седьмой</summary>

Соберём сегодняшние ловушки и посмотрим, что изменилось относительно трёх предыдущих занятий.

**A · данных.** Дубликаты при слиянии: не ошибка, а согласие систем, и схлопывать их надо
сложением вкладов, а не выбрасыванием.

**B · метрики.** Полноту нельзя сравнивать у выдач разной длины — каскад недели 7 возвращает
двадцать пять документов, и его Recall@100 равен Recall@25.

**C · интерпретации.** Три штуки, и все про одно: «RRF значимо лучше BM25» — правда, ничего
не говорящая о пользе слияния; расширение запроса не бесплатно; оптимизм подгонки не требует
злого умысла.

**D · инструмента.** Заполнитель для отсутствующих документов: ноль после z-нормировки означает
«средний», а не «плохой». Метрика не проверяет свои входы и молча выдаёт больше единицы.

**E · замера.** Разделив выборку пополам, мы вдвое ухудшили разрешение обеих оценок.

**F · переноса.** `k = 60` в RRF — константа из статьи 2009 года, а не свойство мира.

**Седьмой тип, которого нет в таблице курса.** Сегодня появился класс ошибок, который не про
данные, не про метрику и не про инструмент: **ошибки процедуры выбора**. Подобрал на тех же
данных, где мерил. Перебрал сто конфигураций вместо десяти. Сравнил с удобной системой вместо
сильнейшей. Ни одна из них не видна в отдельной ячейке; все три видны только в описании того,
**как** ты пришёл к числу.

Отсюда правило занятия: **отчёт о результате обязан описывать процедуру, а не только число.**
Сколько конфигураций перебрано, на чём подбиралось, на чём мерилось, с чем сравнивалось.
Без этих четырёх строк число не проверяемо — а непроверяемое число не результат.
</details>

In [ ]:
ceil_bm = statistics.mean(recall_at(BM[i], GOLD[i], K) for i in range(NQ))
ceil_bi = statistics.mean(recall_at(BI[i], GOLD[i], K) for i in range(NQ))
ceil_union = statistics.mean(
    1.0 if GOLD[i] in set(BM[i][:K]) | set(BI[i][:K]) else 0.0 for i in range(NQ))
only_bm = sum(1 for i in range(NQ)
              if GOLD[i] in set(BM[i][:K]) and GOLD[i] not in set(BI[i][:K]))
only_bi = sum(1 for i in range(NQ)
              if GOLD[i] in set(BI[i][:K]) and GOLD[i] not in set(BM[i][:K]))

print(f"Recall@{K}: BM25 {ceil_bm:.4f} · би-энкодер {ceil_bi:.4f}")
print(f"ПОТОЛОК ОБЪЕДИНЕНИЯ: {ceil_union:.4f}")
print(f"  запас над лучшей одиночной системой: {ceil_union - max(ceil_bm, ceil_bi):+.4f}")
print()
print(f"нашёл ТОЛЬКО BM25:        {only_bm} запросов")
print(f"нашёл ТОЛЬКО би-энкодер:  {only_bi} запросов")
print("это и есть весь материал, из которого слияние может что-то извлечь")
RUN["ceiling_union"], RUN["only_bm"], RUN["only_bi"] = ceil_union, only_bm, only_bi

**Что видно.** Потолок объединения выше лучшей одиночной системы, и последние две строки
объясняют, за счёт чего именно: есть запросы, где документ нашёл только лексический поиск,
и запросы, где только плотный. Сравнивать надо не потолок с единицей, а **запас над лучшей
одиночной**: он и есть весь материал для слияния. Механизм ровно тот, что описан в первом
свёрнутом блоке — у методов разные типы промахов, и множества найденного пересекаются лишь
частично. Чего этот замер НЕ показывает: что слияние этот запас **возьмёт**. Потолок
ограничивает сверху и ничего не обещает — ровно как потолок переранжирования на неделе 7,
до которого мы тогда не дотянули трети. Что делать: считать этот запас первым делом.
Если он около нуля, гибрид не нужен, и это выясняется за одну ячейку вместо дня работы.

<details><summary>Асимметрия находок: почему «только лексический» встречается реже</summary>

Числа последних двух строк несимметричны, и это не случайность нашего корпуса, а свойство
конструкции. Разобраться в нём полезнее, чем в самом слиянии.

**Почему плотный находит больше уникального.** Псевдозапрос — это предложение, вырезанное
из документа. Плотный энкодер сравнивает его с документом целиком и ловит тематическое
соответствие даже там, где конкретные слова разошлись. Лексический нуждается в общих словах;
если предложение было о «наводнении», а остаток документа говорит «затопление» и «уровень воды»,
BM25 промахнётся, а энкодер нет.

**Почему «только лексический» всё-таки не ноль.** Есть класс запросов, где решают редкие
конкретные токены: имена, числа, аббревиатуры, опечатки. Энкодер их размывает — редкий токен
для него почти шум, — а BM25 находит по точному совпадению, и `idf` даёт этому огромный вес.
Именно эти запросы и составляют уникальный вклад лексики.

**Что из этого следует для веса.** Раз уникальный вклад лексики мал, оптимальная `alpha`
должна быть небольшой — и она такой и оказалась. Но это вывод про **наши** запросы. Смести
распределение запросов в сторону артикулов и номеров ошибок, и картина перевернётся:
уникальный вклад лексики вырастет, а вместе с ним и оптимальный вес.

**Практическая проверка, которую стоит делать.** Разбей запросы на классы (короткие и длинные,
с числами и без, с редкими токенами и без) и посчитай уникальный вклад каждой системы **внутри
класса**. Часто выясняется, что глобальная `alpha` — это среднее по двум противоположным
режимам, и правильный ответ не одно число, а маршрутизация: короткие запросы с редкими
токенами — в лексический поиск, длинные тематические — в плотный.
</details>

<details><summary>Потолок объединения: полная формула и что делать с каждым исходом</summary>

Мы посчитали потолок для двух систем. Обобщение и три сценария, которые из него следуют.

**Формула.** Для `n` систем с глубинами `d₁ … dₙ` потолок полноты равен доле запросов, где
правильный ответ попал хотя бы в один из наборов кандидатов. Никакое слияние — ни RRF,
ни обучаемое, ни LLM-переранжирование — не может его превысить. Это самая дешёвая и самая
недооценённая диагностика в поиске: одна строка кода до недели работы.

**Сценарий первый: потолок объединения равен потолку лучшей системы.** Значит, вторая система
не находит ничего нового, и слияние бессмысленно. Такое бывает, когда обе системы плотные
и обучены на похожих данных. Правильный ход — не сливать, а искать **непохожую** вторую
систему, и лексический поиск здесь классический выбор именно из-за непохожести.

**Сценарий второй: потолок заметно выше, слияние его не берёт.** Наш случай. Материал есть,
метод его не извлекает. Причины: обрезка по `k` (мы уместили два списка по сто в сто позиций),
неудачная нормировка, неверный вес. Это чинится и стоит того.

**Сценарий третий: потолок высок и слияние к нему близко.** Дальше расти можно только
увеличением глубины или добавлением третьей системы. Проверяется тем же замером с тремя
списками.

**Тонкость про глубину.** Потолок зависит от `k`, на котором ты объединяешь, а не от `k`,
на котором меряешь. Объединив по сто и обрезав результат до ста, ты теряешь часть материала
неизбежно: сто плюс сто минус пересечение всегда больше ста. Хочешь взять весь запас — меряй
полноту на глубине, равной размеру объединения, либо честно признай, что часть потолка
недостижима по построению.
</details>

---

## Часть 2 · Три способа быть «между» — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 2.1 | ColBERT: как отложить взаимодействие? | считаем MaxSim и сверяем с доской |
| 2.2 | SPLADE: как сделать разреженное обучаемым? | считаем веса и сверяем с доской |
| 2.3 | RRF: как слить, не имея общих шкал? | считаем и сверяем с доской |

Между би-энкодером и кросс-энкодером лежит целое семейство. Три его представителя —
на игрушках, где всё считается руками.

### Шаг 2.1 · ColBERT: MaxSim

Документ кодируется заранее, но **потокенно**. На запросе для каждого токена запроса берётся
максимально похожий токен документа, и всё складывается. Взаимодействие есть, но отложено
на самый конец — и потому дёшево.

In [ ]:
CB = json.load(open(f"{DATA_DIR}/l8-colbert.json", encoding="utf-8"))["toy"]

def maxsim(sim_matrix):
    return sum(max(row) for row in sim_matrix)

rel, irr = CB["docRel"], CB["docIrr"]
got_rel, got_irr = maxsim(rel["sim"]), maxsim(irr["sim"])
assert abs(got_rel - rel["maxSim"]) < 1e-6, "MaxSim релевантного разошёлся с лекцией"
assert abs(got_irr - irr["maxSim"]) < 1e-6, "MaxSim нерелевантного разошёлся с лекцией"
assert [max(r) for r in rel["sim"]] == rel["rowMax"], "построчные максимумы разошлись"

print("токены запроса:", CB["qTokens"])
print(f"\n{'документ':>32} {'построчные максимумы':>26} {'MaxSim':>8}")
for d, got in ((rel, got_rel), (irr, got_irr)):
    print(f"{d['text']:>32} {str([round(max(r), 2) for r in d['sim']]):>26} {got:>8.2f}")
print("\nсверка с data/l8-colbert.json: 3 значения совпали")

**Что видно.** У релевантного документа MaxSim заметно выше, и решают это построчные максимумы:
каждый токен запроса нашёл себе пару. Сравнивать надо не суммы между собой, а **по какой строке
они набраны**: у нерелевантного документа одна строка даёт очень высокое значение — слово `bank`
совпало с `bank` почти идеально, — но остальные две почти пусты. Механизм в этом и состоит:
MaxSim награждает документ, покрывающий **все** аспекты запроса, а не один. Чего эта игрушка
НЕ показывает: цены. Потокенное хранение — это вектор на каждый токен, то есть индекс
в сотню раз больше, чем у обычного би-энкодера, и вся инженерия ColBERT про то, как это сжать.
Что делать: запомнить, что «позднее взаимодействие» покупает точность памятью, а не временем.

<details><summary>Что стоит позднее взаимодействие: арифметика памяти</summary>

MaxSim выглядит бесплатным улучшением: та же предвычислимость, больше точности. Цена есть,
и она в памяти.

**Считаем.** Обычный би-энкодер: один вектор на документ, 384 числа. ColBERT: вектор
на каждый токен. Средний документ в нашем корпусе — около двухсот токенов, значит двести
векторов вместо одного. При том же float32 это двести раз больше — а на реальных корпусах
средняя длина пассажа порядка ста токенов, то есть примерно стократный рост.

**Что с этим делают.** ColBERTv2 сжимает векторы остаточным квантованием: вместо 384 float32
хранится центроид плюс несколько бит на измерение. Получается сокращение примерно
в двадцать-тридцать раз при почти неизменном качестве. Плюс PLAID — движок, который сначала
отбирает документы по центроидам и только для выживших считает полный MaxSim.

**Итог по порядкам.** Голый ColBERT примерно в сто раз тяжелее би-энкодера, ColBERTv2 с PLAID —
примерно в три-пять раз. Это уже приемлемо, и именно поэтому позднее взаимодействие вышло
из разряда академических курьёзов.

**Когда оно окупается.** Когда запросы многоаспектны (несколько независимых требований)
и когда корпус не настолько велик, чтобы память стала главным ограничением. Для поиска
по документации компании — да. Для веб-масштаба — обычно нет, там побеждают дистиллированные
би-энкодеры плюс переранжирование.

**Чего наша игрушка не показывает.** Она работает с готовой матрицей косинусов. В реальности
эту матрицу надо посчитать: `|q| × |d|` скалярных произведений на каждую пару. При глубине
переранжирования сто и документе в двести токенов это шестьдесят тысяч произведений
на запрос — быстро на GPU, заметно на CPU.
</details>

### Шаг 2.2 · SPLADE: разреженное, но обучаемое

BM25 хранит слова, которые в документе есть. SPLADE хранит слова, которые в документе **должны
были бы быть** — с обучёнными весами. Индекс остаётся инвертированным, а веса приходят от модели.

In [ ]:
SP = json.load(open(f"{DATA_DIR}/l8-splade.json", encoding="utf-8"))["toy"]

def splade_weights(logits):
    return [round(math.log(1 + max(0.0, x)), 4) for x in logits]

got = splade_weights(SP["query"]["logits"])
assert got == SP["query"]["weights"], f"веса разошлись с лекцией: {got}"

qw = dict(zip(SP["vocab"], SP["query"]["weights"]))
dw = dict(zip(SP["vocab"], SP["doc"]["weights"]))
dot = sum(qw[t] * dw[t] for t in SP["vocab"])

print("словарь:", SP["vocab"])
print(f"запрос {SP['query']['text']!r} -> логиты {SP['query']['logits']}")
print(f"{'термин':>8} {'вес запроса':>13} {'вес документа':>15} {'вклад':>8}")
for t in SP["vocab"]:
    if qw[t] or dw[t]:
        print(f"{t:>8} {qw[t]:>13.4f} {dw[t]:>15.4f} {qw[t] * dw[t]:>8.4f}")
print(f"\nразреженное скалярное произведение = {dot:.4f}")
print(f"РАСШИРЕНИЕ запроса моделью: {SP['query']['expansion']} "
      f"-- этих слов в запросе НЕТ, а веса у них ненулевые")

**Что видно.** В запросе было два слова, а ненулевых весов — четыре: модель дописала `bank`
и `water`. Сравнивать надо не веса между собой, а **список ненулевых с исходным запросом**:
разница и есть расширение, выученное моделью. Механизм: `log(1 + ReLU(logit))` обнуляет всё
отрицательное и сжимает положительное, оставляя разреженный вектор, который ложится
в инвертированный индекс как обычные термины. Чего эта игрушка НЕ показывает: цены расширения —
чем больше ненулевых координат, тем длиннее постинг-листы и медленнее поиск, и вся регуляризация
SPLADE именно про удержание разреженности. Что делать: заметить, что это тот же приём, что мы
обсуждали в задании про синонимы на неделе 3, — только веса не придуманы, а выучены.

<details><summary>SPLADE: почему разреженность приходится удерживать силой</summary>

Модель, которой разрешили приписывать вес любому слову словаря, естественным образом припишет
ненулевой вес почти всему. Разреженность у SPLADE — не свойство архитектуры, а результат
специального давления.

**Откуда берётся плотность.** Выход — логиты по всему словарю BERT, тридцать тысяч позиций.
`ReLU` обнуляет отрицательные, но обученная без ограничений модель быстро выучивает, что
чуть-чуть положительный вес почти везде повышает полноту. Получается вектор с тысячами
ненулевых координат, и инвертированный индекс перестаёт быть быстрым: постинг-листы раздуваются,
а запрос трогает половину корпуса.

**Как удерживают.** Регуляризатор FLOPS — оценка ожидаемого числа операций при поиске,
добавленная прямо в функцию потерь. Модель платит за каждую ненулевую координату, и цена
подобрана так, чтобы получалось порядка сотни-двух ненулевых на документ. Это компромисс
«качество против скорости», зашитый в обучение, а не выбираемый при поиске.

**Практическое следствие.** У SPLADE есть параметр, которого нет у BM25: **сколько ты готов
заплатить за полноту**. Опубликованные варианты (SPLADE, SPLADE++, дистиллированные версии)
отличаются в том числе точкой на этом компромиссе. Выбирая модель, надо смотреть не только
на метрику, но и на среднее число ненулевых координат — оно определяет скорость поиска
на твоём железе.

**Связь с сегодняшним занятием.** SPLADE — это гибрид, встроенный в одну модель: разреженный
индекс, как у лексического поиска, и семантическое расширение, как у плотного. То есть
альтернатива тому, чем мы занимались руками в частях 3 и 4. Плата за неё — обучение и потеря
контроля: соотношение лексического и семантического зашито в веса, и `alpha` подкрутить нельзя.
</details>

⚠️ Ловушка C · **Расширение запроса не бесплатно.** SPLADE дописывает термины и тем самым
поднимает полноту — и ровно так же поднимает шанс притащить нерелевантное. На неделе 3 мы
разбирали это словами; здесь тот же компромисс зашит в обучение, и регуляризатор решает, где
остановиться. Он подобран под MS MARCO, а не под твой корпус.

### Шаг 2.3 · RRF: слияние без общих шкал

Reciprocal rank fusion складывает **не скоры, а обратные ранги**: `1/(k + позиция)`. Шкалы
систем при этом не нужны вовсе — сравниваются только места.

In [ ]:
H = json.load(open(f"{DATA_DIR}/l8-hybrid.json", encoding="utf-8"))

def rrf(lists, k=RRF_K, top=None):
    sc = defaultdict(float)
    for lst in lists:
        for rank, doc in enumerate(dedupe(lst), 1):
            sc[doc] += 1.0 / (k + rank)
    order = sorted(sc, key=lambda d: (-sc[d], d))
    return (order[:top] if top else order), dict(sc)

order, sc = rrf([H["sparse"]["order"], H["dense"]["order"]], k=H["k"])
expected = [f["id"] for f in H["fused"]]
assert order == expected, f"порядок RRF разошёлся с лекцией: {order} против {expected}"
for f in H["fused"]:
    assert abs(sc[f["id"]] - f["score"]) < 1e-4, f"скор {f['id']} разошёлся"

print(f"лексический порядок: {H['sparse']['order']}")
print(f"плотный порядок:     {H['dense']['order']}")
print(f"\n{'док':>5} {'ранг лекс.':>12} {'ранг плотн.':>13} {'RRF':>9}")
for f in H["fused"]:
    print(f"{f['id']:>5} {f['rSparse']:>12} {f['rDense']:>13} {sc[f['id']]:>9.4f}")
print(f"\nсверка с data/l8-hybrid.json: порядок и {len(H['fused'])} скоров совпали")

**Что видно.** Победил не тот документ, который первый у лексического поиска, и не тот, который
первый у плотного, — победил **D2, второй у одного и первый у другого**. Сравнивать надо
не итоговые скоры, а **пару рангов у каждой строки**: D1 стоит первым у лексического
и последним у плотного, и суммарно проигрывает согласованному D2. Механизм и есть смысл RRF:
вклад каждой системы **ограничен сверху** величиной `1/(k+1)`, поэтому первое место нельзя
«докупить» — оно стоит не больше 1/61, — а плохой ранг в другом списке приносит почти столько же,
сколько средний. При `k=60` кривая на верхних местах почти плоская, и сумма вкладов близко следует
**сумме рангов**: у D2 она равна 3, у D1 — 6, отсюда 0,0325 против 0,0318. Заметь, что дело
именно в ограниченности вклада, а не в вогнутости: `1/(k+r)` **выпукла** (`f''(r) = 2/(k+r)³ > 0`),
и при одинаковой сумме рангов края дали бы чуть больше середины. Чего эта игрушка НЕ показывает: что будет,
если системы неравноценны, — RRF считает их голоса равными по построению, и слабую систему
приходится либо взвешивать, либо не включать. Что делать: помнить, что RRF — это голосование,
а не измерение.

<details><summary>RRF или слияние скоров: решающее дерево на четыре вопроса</summary>

Выбор между двумя подходами не вопрос вкуса. Он определяется четырьмя свойствами твоей
задачи, и ответить на них можно до всякой реализации.

**Вопрос первый: скоры калиброваны?** Если да — складывай величины, калибровка стоила денег
и выбрасывать её глупо. Если нет (обычный случай) — переходи ко второму.

**Вопрос второй: сколько у тебя размеченных запросов?** Слияние скоров требует подобрать
как минимум `alpha`, нормировку и заполнитель — это три гиперпараметра. Мы видели в части 4,
что уже одиннадцать конфигураций на сорока запросах дают заметный оптимизм. Если запросов
меньше сотни, RRF почти наверняка честнее: у него подбирать нечего. Если тысячи — можно
позволить себе слияние скоров и подобрать всё как следует.

**Вопрос третий: системы сопоставимы по силе?** RRF считает голоса равными. Если одна система
заметно слабее, её голос всё равно весит столько же, и итог портится. Тут либо взвешенный RRF
(`w₁/(k+r₁) + w₂/(k+r₂)`), либо слияние скоров, либо просто выбросить слабую. Проверяется это
тем же потолком объединения: если уникальный вклад одной системы близок к нулю, она не нужна.

**Вопрос четвёртый: стабильны ли скоры во времени?** Если модель дообучается или подменяется,
скоры уезжают, и подобранная `alpha` устаревает молча. Ранги устойчивее: они переживают любое
монотонное преобразование скора. В системах, где модель обновляется еженедельно, это весомый
аргумент за RRF.

**Практический вывод.** На старте проекта — RRF: он честен при малой разметке, не требует
подбора и не устаревает. Когда накопятся тысячи размеченных запросов и модель стабилизируется —
имеет смысл попробовать слияние скоров и **честно** проверить, окупается ли выигрыш добавленной
сложностью. Часто оказывается, что нет.

**И то, о чём забывают.** Оба метода — частные случаи обучаемого слияния с одним признаком.
Как только признаков становится больше двух (свежесть, популярность, качество источника,
клики), выбор исчезает: нужен LTR, и вопрос переходит в плоскость «какие признаки», а не
«как сложить два числа».
</details>

<details><summary>Почему RRF работает лучше, чем имеет право</summary>

RRF — метод в одну строку без единого обучаемого параметра, и он регулярно обыгрывает
обученные комбинации. Это выглядит несправедливо и имеет объяснение.

**Первое: он не может переобучиться.** У него нет параметров, подгоняемых под данные (кроме
`k`, к которому результат малочувствителен). Всё, что мы измеряли в части 4 как оптимизм,
для RRF равно нулю по построению. Обученная комбинация выигрывает на данных, где обучалась,
и теряет часть выигрыша на новых; RRF не выигрывает нигде и не теряет ничего.

**Второе: ранги устойчивее скоров.** Скор системы может съехать от смены версии модели,
длины запроса, состава корпуса. Ранг — порядковая величина, и она инвариантна к любому
монотонному преобразованию скора. Система, у которой скоры уехали вдвое, даёт те же ранги
и тот же вклад в RRF.

**Третье: форма кривой.** Функция `1/(k+r)` убывает медленно и **выпукла** — она круче всего
на верхних местах. Это означает, что
переход с пятого места на первое ценится дороже, чем со сто пятого на сто первое, — то есть
форма примерно соответствует убывающему вниманию пользователя, тому же, что моделирует
дисконт nDCG. Совпадение не случайное: и то и другое подобрано под одно и то же поведение.

**Где он проигрывает.** Там, где скоры **калиброваны** и потому несут информацию сверх порядка.
Если система умеет сказать «этот документ релевантен с вероятностью 0,95, а следующий — 0,3»,
выбрасывать эту разницу жалко. В индустрии калиброванные скоры редки именно потому, что
калибровка требует разметки и переобучения, — и потому RRF так часто оказывается практичным
выбором по умолчанию.

**Что стоит унести.** Метод без параметров — не признак примитивности. Отсутствие параметров
это отсутствие способа себя обмануть, и в задачах с маленькой выборкой оценки это преимущество
перевешивает потенциальный выигрыш обученного метода.
</details>

⚠️ Ловушка F · **`k = 60` — не универсальная константа.** Это значение из работы Cormack,
Clarke и Buettcher 2009 года, подобранное на их коллекциях. Оно управляет тем, насколько сильно
верхние места важнее нижних: при малом `k` первое место доминирует, при большом различия между
местами сглаживаются. Переносить 60 как «правильное число» нельзя — это такой же подбираемый
параметр, как `b` в BM25.

---

## Часть 3 · Слияние на своих данных — 30 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 3.1 | Что даёт RRF на нашем корпусе? | меряем и впервые за курс получаем различимый эффект |
| 3.2 | Как сложить скоры, если шкалы разные? | две нормировки, и они дают разный ответ |
| 3.3 | Какой вес лучше? | прогон по alpha на двух метриках сразу |

In [ ]:
RRF_RUN = [rrf([BM[i][:K], BI[i][:K]], top=K)[0] for i in range(NQ)]

def report(name, orders):
    return (name,
            statistics.mean(rr_at(o, g) for o, g in zip(orders, GOLD)),
            statistics.mean(recall_at(o, g, K) for o, g in zip(orders, GOLD)))

rows = [report("BM25", BM), report("би-энкодер", BI), report("RRF", RRF_RUN)]
print(f"{'система':>14} {'MRR@10':>9} {'Recall@' + str(K):>11}")
for n, m, r in rows:
    print(f"{n:>14} {m:>9.4f} {r:>11.4f}")
print(f"{'потолок объед.':>14} {'--':>9} {ceil_union:>11.4f}")
RUN["rrf"] = {"mrr": rows[2][1], "recall": rows[2][2]}

**Что видно.** RRF по полноте догнал лучшую из двух систем и **не превзошёл** её, оставшись
заметно ниже потолка объединения. Сравнивать надо не RRF с BM25, а **RRF с би-энкодером
и с потолком**: запас, который мы посчитали в части 1, взят не был. Механизм понятен, если
вспомнить, что мы режем результат на глубине сто: RRF обязан уместить в сто позиций документы
из двух списков по сто, и часть неизбежно выпадает. Чего эта таблица НЕ показывает: различимы
ли различия — следующая ячейка, и там впервые за курс будет «да». Что делать: не радоваться
строке «RRF лучше BM25». Она верна и почти бессодержательна: RRF унаследовал полноту
би-энкодера, а не создал новую.

In [ ]:
def paired(a, b, label):
    d = [x - y for x, y in zip(a, b)]
    m = statistics.mean(d)
    se = statistics.stdev(d) / math.sqrt(len(d))
    lo, hi = m - 2.0 * se, m + 2.0 * se
    verdict = "РАЗЛИЧИМО" if lo > 0 or hi < 0 else "не отличимо"
    print(f"  {label:32} Δ={m:+.4f} ДИ95≈[{lo:+.4f}; {hi:+.4f}] "
          f"{sum(x > 0 for x in d):>2}/{sum(x < 0 for x in d):<2} {verdict}")
    return {"delta": m, "ci": [lo, hi], "distinguishable": lo > 0 or hi < 0}

mq = lambda O: [rr_at(o, g) for o, g in zip(O, GOLD)]
rq = lambda O: [recall_at(o, g, K) for o, g in zip(O, GOLD)]
print("MRR@10:")
p1 = paired(mq(RRF_RUN), mq(BM), "RRF против BM25")
p2 = paired(mq(RRF_RUN), mq(BI), "RRF против би-энкодера")
print(f"Recall@{K}:")
p3 = paired(rq(RRF_RUN), rq(BM), "RRF против BM25")
p4 = paired(rq(RRF_RUN), rq(BI), "RRF против би-энкодера")
RUN["paired_rrf"] = {"mrr_vs_bm25": p1, "mrr_vs_bi": p2, "rec_vs_bm25": p3, "rec_vs_bi": p4}

**Что видно.** Впервые за курс одно сравнение оказалось **различимым**: RRF против BM25
по полноте, интервал целиком правее нуля, и ни одного проигранного запроса. Сравнивать надо
именно четыре строки между собой, а не каждую с нулём: три «не отличимо» и одно «различимо» —
и различимым оказалось не то, что мы хотели показать. Механизм честный и слегка обидный: RRF
выигрывает у BM25 по полноте, потому что тащит в себе би-энкодер, а против самого би-энкодера
даёт ровно ноль. То есть различимый эффект есть, синергии нет. Чего этот результат НЕ показывает:
что гибрид бесполезен — мы пока сливали только рангами и только на одной глубине. Что делать:
запомнить форму вывода. «Различимо» относится к **конкретной паре систем на конкретной метрике**,
и переносить это слово на всю строку «гибрид работает» нельзя.

<details><summary>Как читать четыре строки парных сравнений, не обманув себя</summary>

Мы напечатали четыре сравнения, одно из них различимо. Это ровно та ситуация, где родится
неверный вывод, если читать невнимательно.

**Ошибка первая: взять различимое и забыть остальные.** «RRF статистически значимо лучше
BM25» — правда. Но мы проверили четыре гипотезы, и при четырёх проверках вероятность хотя бы
одного ложного «значимо» при пороге 0,05 близка к восемнадцати процентам. Формально следовало
бы поправить порог (Бонферрони поделил бы 0,05 на четыре). Наш случай спасает то, что интервал
далеко от нуля, а не то, что мы аккуратны.

**Ошибка вторая: считать «не отличимо» доказательством равенства.** Три строки говорят
«не отличимо от нуля», и это значит только, что наших данных не хватает для различения.
Интервал `[-0,05; +0,10]` совместим и с нулём, и с приростом в десять пунктов. Формулировка
«разницы нет» здесь неверна; верная — «наш замер не обладает разрешением».

**Ошибка третья: сравнивать с удобным.** Различимым оказалось сравнение с **худшей** системой.
Это почти всегда так: чем слабее база, тем легче показать значимость. Отсюда правило чтения
чужих работ — первым делом смотреть, с чем сравнивали, и есть ли в таблице результат сильнейшего
доступного метода.

**Как надо было бы.** Зафиксировать **одно** главное сравнение до эксперимента — например,
«гибрид против лучшей одиночной системы по MRR@10» — и остальные объявить разведочными.
Тогда порог не требует поправки, а разведочные числа честно называются разведочными и служат
поводом для следующего эксперимента, а не выводом.
</details>

⚠️ Ловушка C · **«RRF статистически значимо лучше BM25» — правда, которая вводит в заблуждение.**
Утверждение верно и проверено. Оно ничего не говорит о пользе слияния: тот же выигрыш даёт
просто взять би-энкодер и выбросить BM25. Всегда сравнивай гибрид с **лучшей** из сливаемых
систем, а не с худшей.

### Шаг 3.2 · Сложить скоры: сначала привести к общей шкале

RRF выбрасывает величины и оставляет места. Второй путь — сохранить величины, но привести
их к сопоставимым единицам. Способов нормировки два, и они дают **разный** ответ.

In [ ]:
def zscore(d):
    if not d:
        return {}
    v = np.fromiter(d.values(), float)
    mu, sd = v.mean(), v.std()
    return {k: ((x - mu) / sd if sd > 0 else 0.0) for k, x in d.items()}

def minmax(d):
    if not d:
        return {}
    lo, hi = min(d.values()), max(d.values())
    return {k: ((x - lo) / (hi - lo) if hi > lo else 0.0) for k, x in d.items()}

def fuse(i, alpha, norm="z"):
    lex = dict(zip(BM[i][:K], BMS[i][:K]))
    dense = dict(zip(BI[i][:K], BIS[i][:K]))
    f = zscore if norm == "z" else minmax
    lex, dense = f(lex), f(dense)
    floor = -4.0 if norm == "z" else 0.0     # чем заполнять отсутствие в чужом списке
    cand = set(lex) | set(dense)
    sc = {d: alpha * lex.get(d, floor) + (1 - alpha) * dense.get(d, floor) for d in cand}
    return sorted(sc, key=lambda d: (-sc[d], d))[:K]

probe = 0
lex_raw = dict(zip(BM[probe][:5], BMS[probe][:5]))
print("сырые скоры BM25 (топ-5 первого запроса):",
      {d: round(v, 2) for d, v in lex_raw.items()})
print("после z-нормировки: ", {d: round(v, 2) for d, v in zscore(lex_raw).items()})
print("после min-max:      ", {d: round(v, 2) for d, v in minmax(lex_raw).items()})

**Что видно.** Одни и те же пять чисел после двух нормировок выглядят по-разному: z-оценка
центрирует по среднему и оставляет отрицательные значения, min-max растягивает на отрезок
от нуля до единицы и всегда даёт ровно один ноль и ровно одну единицу. Сравнивать надо
не сами наборы, а **что каждая нормировка сохраняет**: z-оценка сохраняет форму распределения
и потому чувствительна к выбросам меньше, min-max привязывается к двум крайним значениям
и потому от единственного выброса зависит целиком. Механизм важен для слияния: min-max
принудительно делает лучший документ каждого списка равным единице, то есть **стирает
информацию о том, насколько уверена система**. Чего этот вывод НЕ показывает: какая нормировка
лучше — это зависит от данных, и ниже мы увидим, что они дают разный оптимум. Что делать:
всегда печатать скоры до и после, а не доверять функции на слово.

<details><summary>Что такое «калиброванный скор» и почему его почти ни у кого нет</summary>

В разборе несколько раз всплывало слово «калибровка». Оно стоит отдельного объяснения, потому
что оно объясняет, почему индустрия так часто выбирает RRF.

**Определение.** Скор калиброван, если его величина означает одно и то же независимо
от запроса. Строгая формулировка: среди всех документов со скором `s` доля релевантных
равна `f(s)` для некоторой фиксированной функции `f`, одной на все запросы. В идеале `f`
тождественна, и скор прямо равен вероятности релевантности.

**Почему это ценно.** Калиброванный скор позволяет три вещи, невозможные без него: ставить
порог отсечения («не показывать ниже 0,3»), складывать скоры разных систем без нормировки
и принимать решения о показе вообще (пустая выдача лучше плохой).

**Почему его почти нет.** Обучение ранжированию оптимизирует **порядок**, и функция потерь
безразлична к абсолютным значениям: сдвинь все скоры на константу — порядок не изменится,
потери не изменятся, метрика не изменится. Модель поэтому не имеет никакого стимула делать
скоры осмысленными по величине, и они таковыми не оказываются.

**Как калибруют.** Отдельным шагом после обучения: берут размеченные пары, строят монотонное
преобразование скора в вероятность — Платт (логистическая регрессия по одному признаку)
или изотоническая регрессия (произвольная монотонная функция). Требуется разметка, и калибровку
надо переобучать при каждом изменении модели или заметном сдвиге распределения запросов.

**Как проверить, калиброван ли твой скор.** Разбей документы по скору на десять корзин,
в каждой посчитай долю релевантных, нарисуй график «средний скор корзины против доли
релевантных». У калиброванной системы точки лягут на диагональ. У некалиброванной — на что
угодно, чаще всего на кривую, задранную вверх. Это десять строк кода и мгновенно закрывает
вопрос, можно ли складывать скоры напрямую.

**Связь с сегодняшним.** Мы нормировали скоры внутри каждого запроса именно потому, что они
не калиброваны. Будь они калиброваны, часть 3 свелась бы к одной строке сложения без всяких
`alpha` и заполнителей.
</details>

<details><summary>Ещё три нормировки, которые применяют в проде</summary>

Мы взяли две самые простые. В реальных гибридах встречаются ещё как минимум три, и у каждой
своя логика.

**Нормировка на максимум (без вычитания минимума).** `x / max(x)`. Сохраняет ноль как ноль,
что важно, когда скор ноль означает «термин не встретился». Для BM25 это осмысленно, для
косинуса — нет: у косинуса ноль означает «ортогонально», то есть вполне определённое, а не
отсутствие.

**Логарифмическая.** `log(1 + x)` перед любой другой нормировкой. Применяют, когда у скоров
тяжёлый правый хвост: один документ с огромным BM25 иначе задавит всех. У нас хвост умеренный,
и разница была бы невелика.

**Ранговая с сохранением величины.** Заменить скор на его перцентиль внутри выдачи. Даёт
равномерное распределение на отрезке, устойчиво к любым выбросам и при этом сохраняет больше
информации, чем чистые ранги RRF, — потому что перцентиль учитывает, сколько документов
имеет скор ниже. Хороший компромисс, применяется редко просто по традиции.

**Как выбирать.** Единственный честный способ — прогнать каждую нормировку
на отложенных данных и сравнить, что мы отчасти и сделали для двух. Но заметь: перебирая
нормировки, ты увеличиваешь число конфигураций, и оптимизм из части 4 растёт вместе с ним.
Три нормировки по одиннадцать весов — это тридцать три конфигурации, и выбирать среди них
на сорока запросах уже опасно.

**Что стоит унести.** Нормировка — не техническая деталь перед «настоящим» слиянием.
Это полноценный гиперпараметр, и относиться к нему надо так же: фиксировать заранее либо
подбирать честно, с отложенными данными.
</details>

⚠️ Ловушка D · **Чем заполнять отсутствие?** Документ из списка BM25 может не встречаться
у би-энкодера, и его плотный скор неизвестен. Ноль здесь — **не** нейтральное значение: после
z-нормировки ноль означает «средний документ», то есть отсутствующему приписывается вполне
приличная оценка. Мы ставим `−4` для z и `0` для min-max, то есть «хуже всех, кого мы видели»,
и это тоже выбор, а не истина. Третий вариант — считать полные скоры для объединения кандидатов,
но это требует прогнать обе системы по всему объединению и стоит времени.

In [ ]:
sweep = {}
for norm in ("z", "minmax"):
    sweep[norm] = {}
    for a in ALPHAS:
        F = [fuse(i, a, norm) for i in range(NQ)]
        sweep[norm][a] = {
            "mrr": statistics.mean(rr_at(o, g) for o, g in zip(F, GOLD)),
            "recall": statistics.mean(recall_at(o, g, K) for o, g in zip(F, GOLD)),
        }

print(f"{'alpha':>6} {'MRR (z)':>9} {'Recall (z)':>12} {'MRR (minmax)':>14} {'Recall (mm)':>13}")
for a in ALPHAS:
    z, m = sweep["z"][a], sweep["minmax"][a]
    print(f"{a:>6} {z['mrr']:>9.4f} {z['recall']:>12.4f} {m['mrr']:>14.4f} {m['recall']:>13.4f}")
best = {n: {k: max(ALPHAS, key=lambda a: sweep[n][a][k]) for k in ("mrr", "recall")}
        for n in sweep}
print(f"\nлучшая alpha по MRR:    z -> {best['z']['mrr']} · min-max -> {best['minmax']['mrr']}")
print(f"лучшая alpha по Recall: z -> {best['z']['recall']} · min-max -> {best['minmax']['recall']}")
RUN["sweep"] = {n: {str(a): v for a, v in d.items()} for n, d in sweep.items()}

**Что видно.** Смена нормировки двигает оптимум сильнее, чем смена метрики. Сравнивать надо
не столбцы попарно, а **положение максимума в каждом**: под z-оценкой обе метрики согласны
и указывают на один вес, под min-max они расходятся между собой и обе уезжают от z-ответа.
То есть «лучший вес» — свойство не данных, а всей связки «нормировка плюс метрика плюс глубина»,
и назвать его, не назвав связку, нельзя. Механизм в предыдущем разборе: min-max стирает
уверенность системы, приравнивая лучший документ каждого списка к единице, и потерянный вклад
приходится компенсировать весом. Чего эта таблица НЕ показывает: устойчивости максимумов —
соседние значения `alpha` отличаются на сотые, а это уже похоже на шум, который мы много раз
видели. Что делать: **не выбирать alpha по этой таблице.** Она посчитана на всех запросах сразу,
и следующая часть показывает, чего стоит такой выбор.

<details><summary>Одна alpha на все запросы — или маршрутизация?</summary>

Мы подбирали единственный вес для всех запросов. Это самое грубое из возможных решений,
и стоит понимать, что находится дальше.

**Наблюдение, из которого всё следует.** Оптимальный вес почти наверняка разный для разных
типов запросов. Запрос «нхл плейофф 1993» лучше обслуживается лексикой, запрос «как устроено
торможение спускаемого аппарата» — плотным поиском. Единая `alpha` — это компромисс между
двумя режимами, и она хуже обоих на их территории.

**Маршрутизация.** Простейший вариант: классификатор, который по запросу решает, какую систему
использовать или какой вес взять. Признаки дешёвые — длина запроса, средний `idf` его терминов,
доля токенов вне словаря, наличие цифр. Обучается на тех же размеченных запросах.

**Адаптивный вес.** Мягче маршрутизации: `alpha` предсказывается как функция признаков запроса,
а не выбирается из двух вариантов. По сути это уже LTR с одним обучаемым коэффициентом,
зависящим от контекста.

**Почему это редко делают.** Потому что для обучения маршрутизатора нужна разметка,
разбитая по классам запросов, и каждого класса должно хватать. При наших восьмидесяти запросах
разговор бессмысленен: на класс приходилось бы по два десятка, и оптимизм отбора съел бы всё.
Маршрутизация окупается на тысячах размеченных запросов, то есть в проде, а не на семинаре.

**Что можно сделать дёшево уже сейчас.** Посчитать оптимальную `alpha` отдельно для двух
половин запросов, разбитых по длине, и посмотреть, различаются ли они. Если различаются сильно —
это аргумент за маршрутизацию, полученный за одну ячейку. Если нет — единая `alpha` оправдана,
и это тоже результат.
</details>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharex=True)
for j, key in enumerate(("mrr", "recall")):
    for norm, style in (("z", "o-"), ("minmax", "s--")):
        ax[j].plot(ALPHAS, [sweep[norm][a][key] for a in ALPHAS], style, label=norm)
    ax[j].set_xlabel("alpha -- вес лексической ступени")
    ax[j].set_title("MRR@10" if key == "mrr" else f"Recall@{K}")
    ax[j].legend(fontsize=9)
ax[0].axhline(rows[0][1], color="#B4521F", ls=":", lw=1)
ax[1].axhline(ceil_union, color="#2E7D52", ls=":", lw=1)
plt.suptitle("alpha=0 -- чистый плотный, alpha=1 -- чистый BM25; пунктир: BM25 и потолок")
plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо не две панели друг с другом, а **форму кривых внутри каждой**:
слева максимум пологий и смещён влево, справа кривая держится почти горизонтально до `alpha`
около половины и потом резко падает. Ожидаемая картина по механизму: полнота держится, пока
плотная система вносит достаточный вклад, и обваливается, когда лексическая её вытесняет,
— а точность в верхушке от этого страдает раньше и мягче. Чего график НЕ показывает:
доверительных интервалов; визуально пик выглядит содержательным, а разница между `alpha=0,2`
и `alpha=0,5` вполне может быть шумом. Что делать: смотреть на **ширину плато**, а не
на положение пика. Плато широкое — значит выбор внутри него безопасен; узкий пик почти всегда
означает подгонку.

---

## Часть 4 · Подбор без подгонки — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 4.1 | Сколько стоит подбор на тех же данных? | делим запросы пополам и меряем оптимизм |
| 4.2 | Как учат веса по-настоящему? | RankNet и LambdaRank на игрушке из лекции |

Мы только что перебрали одиннадцать значений `alpha` и посмотрели на результат. Это ровно
множественное сравнение из недели 4, только вместо гипотез перебирались конфигурации.

In [ ]:
rng = random.Random(SEED)
idx = list(range(NQ))
rng.shuffle(idx)
tune, held = idx[:NQ // 2], idx[NQ // 2:]

def mrr_on(subset, alpha, norm="z"):
    return statistics.mean(rr_at(fuse(i, alpha, norm), GOLD[i]) for i in subset)

alpha_tuned = max(ALPHAS, key=lambda a: mrr_on(tune, a))
alpha_all = max(ALPHAS, key=lambda a: mrr_on(idx, a))

honest = mrr_on(held, alpha_tuned)          # подобрали на одних, померяли на других
optimistic = mrr_on(tune, alpha_tuned)      # ТА ЖЕ alpha, замер там же, где подбирали
optimism = optimistic - honest              # чистая цена подгонки: меняется только набор

print(f"запросов на подбор: {len(tune)} · на проверку: {len(held)}")
print(f"alpha, подобранная на половине-подборе: {alpha_tuned}")
print(f"alpha, подобранная на ВСЕХ запросах:    {alpha_all}")
print()
print(f"ЧЕСТНО   (подбор на одних, замер на других): MRR@10 = {honest:.4f}")
print(f"С ПОДГОНКОЙ (замер там же, где подбирали):   MRR@10 = {optimistic:.4f}")
print(f"ОПТИМИЗМ подгонки: {optimism:+.4f}")
print(f"для масштаба: эффект слияния над би-энкодером (alpha с подбора, все запросы): "
      f"{mrr_on(idx, alpha_tuned) - rows[1][1]:+.4f}")
RUN["alpha_tuned"], RUN["honest_mrr"], RUN["optimism"] = alpha_tuned, honest, optimism

**Что видно.** Оптимизм подгонки оказывается **того же порядка, что весь эффект слияния**:
+0,0445 подгонки при +0,0501 эффекта. Сравнивать надо не два значения MRR, а **разницу между
ними с величиной эффекта**: здесь она почти его целиком и составляет, а мелкая сетка — это
проверит задание 2 — поднимет оптимизм выше самого эффекта. Механизм ровно тот же, что при
множественных сравнениях: перебирая одиннадцать значений и выбирая максимум, ты выбираешь
не лучшее `alpha`, а то, которому больше повезло на этих сорока запросах. Чего этот замер
НЕ показывает: что слияние бесполезно. Эффект +0,0501 померен при честно подобранной `alpha`
на всех запросах и никуда не делся; подгонкой было бы отчитаться числом 0,5275 с половины,
на которой подбирали, вместо 0,4830 с отложенной. Что делать: делить запросы **до** подбора,
а не после, и в отчёте называть число, полученное на отложенной половине. Оно всегда скромнее
и всегда честнее.

<details><summary>Оптимизм отбора: откуда он берётся и как его оценивают правильно</summary>

Мы измерили оптимизм прямым способом — разделив данные. Полезно понимать, откуда он берётся
и что делать, когда делить нечего.

**Природа явления.** Пусть все конфигурации на самом деле одинаковы, а наблюдаемая метрика
равна истинной плюс шум. Выбирая максимум из `m` наблюдений, ты выбираешь конфигурацию
с наибольшим **шумом**, и ожидаемая величина этого максимума растёт примерно как корень
из логарифма `m`, умноженный на стандартное отклонение шума. Отсюда два следствия: оптимизм
растёт с числом конфигураций медленно (логарифмически) и с шумом — линейно. На маленькой
выборке запросов шум велик, и оптимизм становится заметным даже при десятке конфигураций.

**Кросс-валидация вместо разделения.** Разделив восемьдесят запросов пополам, мы ухудшили обе
оценки. Правильнее пятикратная кросс-валидация по запросам: подбор на четырёх пятых, замер
на оставшейся, повторить пять раз, усреднить. Оценка получается почти несмещённой и использует
все данные. Цена — пятикратный счёт, у нас это секунды.

**Вложенная кросс-валидация.** Если подбираешь не только `alpha`, но и, скажем, нормировку,
внешний цикл должен быть отдельным от внутреннего: во внутреннем подбираешь, во внешнем
меряешь. Иначе выбор нормировки просачивается в оценку, и оптимизм возвращается.

**Правило Бутстрапа .632.** Классическая альтернатива, когда данных мало: оценка по бутстрап-
выборкам с поправкой на смещение. В поиске применяется редко, потому что единица наблюдения —
запрос, а запросов обычно и так мало.

**Самое дешёвое, что можно сделать всегда.** Записать число конфигураций, которые ты
попробовал, рядом с результатом. Даже без всякой поправки эта строка позволяет читателю
самому оценить, сколько доверия давать максимуму.
</details>

⚠️ Ловушка C · **Оптимизм не требует злого умысла.** Никто не подтасовывал: мы честно
перебрали сетку и честно взяли максимум. Смещение возникает из самой процедуры выбора,
и единственная защита — отложенные данные, зафиксированные заранее.

⚠️ Ловушка E · **Половина от восьмидесяти — это сорок.** Разделив выборку, мы вдвое ухудшили
разрешение обеих оценок. При маленькой выборке это болезненно, и правильный ответ —
кросс-валидация по запросам: пять разбиений, подбор на четырёх пятых, замер на оставшейся,
среднее по пяти. Мы этого не делаем ради времени, и это названо вслух.

### Шаг 4.2 · Как веса учат по-настоящему

Перебор одного числа работает, пока признак один. При десяти признаках сетка не поможет,
и веса учат. Две идеи из лекции — на игрушке, где всё считается руками.

In [ ]:
LTR = json.load(open(f"{DATA_DIR}/l8-ltr.json", encoding="utf-8"))["toy"]

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

si, sj = LTR["pair"]["docI"]["score"], LTR["pair"]["docJ"]["score"]
diff = si - sj
prob = sigmoid(diff)
cost = -math.log(prob)
grad = 1 - prob

assert abs(diff - LTR["scoreDiff"]) < 1e-9, "разница скоров разошлась с лекцией"
assert abs(prob - LTR["rankNetProb"]) < 1e-4, "вероятность RankNet разошлась"
assert abs(cost - LTR["rankNetCost"]) < 1e-4, "функция потерь RankNet разошлась"
assert abs(grad - LTR["gradient"]) < 1e-4, "градиент разошёлся"
assert abs(grad * LTR["ndcg"]["deltaNdcg"] - LTR["lambda"]) < 1e-4, "лямбда разошлась"

print(f"пара: релевантный {si}, нерелевантный {sj} · разница {diff}")
print(f"RankNet: P(i выше j) = sigmoid({diff}) = {prob:.4f} · потери {cost:.4f} "
      f"· градиент {grad:.4f}")
print(f"LambdaRank: перестановка меняет nDCG на {LTR['ndcg']['deltaNdcg']:.4f} "
      f"({LTR['ndcg']['current']:.4f} -> {LTR['ndcg']['afterSwap']:.4f})")
print(f"            лямбда = градиент x дельта-nDCG = {grad * LTR['ndcg']['deltaNdcg']:.4f}")
print("сверка с data/l8-ltr.json: 5 значений совпали")

**Что видно.** RankNet считает вероятность того, что релевантный документ стоит выше
нерелевантного, и штрафует за неуверенность; LambdaRank домножает этот градиент на то,
**насколько метрика изменится** от перестановки этой пары. Сравнивать надо не потери
с градиентом, а **градиент с лямбдой**: они отличаются множителем дельта-nDCG, и в нём вся
идея. Механизм важен: пара, перестановка которой почти не двигает метрику, получает почти
нулевой вес, и модель не тратит на неё усилий. Именно так решается разрыв между гладкой
функцией потерь и ступенчатой метрикой — не сглаживанием метрики, а взвешиванием пар.
Чего эта игрушка НЕ показывает: как это масштабируется — в реальном LambdaMART деревьев тысячи,
а пар квадратично много, и половина инженерии про их отбор. Что делать: запомнить, что
«обучение ранжированию» — это не «обучить регрессию на релевантность», а обучение **порядку**,
и целевая метрика входит в градиент явно.

<details><summary>Почему обучение ранжированию не сводится к регрессии на релевантность</summary>

Естественная мысль: раз есть разметка, обучим модель предсказывать релевантность, а потом
отсортируем по предсказанию. Мысль работает плохо, и причины поучительны.

**Первая: метрика смотрит на порядок, а не на значения.** Модель, ошибающаяся на 0,3 в оценке
каждого документа одинаково, даёт идеальный порядок и нулевую метрику ошибки регрессии.
Модель, точная везде, кроме одной пары наверху, даёт отличную регрессию и плохой nDCG.
Оптимизируя не то, получаешь не то.

**Вторая: важность позиции.** Ошибка на первом месте стоит дороже ошибки на сотом, а регрессия
считает их равными. LambdaRank решает это буквально — домножает градиент пары на изменение
метрики от их перестановки, и пары из хвоста получают почти нулевой вес.

**Третья: масштаб релевантности не универсален.** Оценка «3» на одном запросе и «3» на другом
не означают одно и то же: на лёгком запросе троек десятки, на трудном ни одной. Ранжирование
внутри запроса от этого не страдает, а регрессия учится смешивать запросы и портится.

**Три семейства подходов.** Поточечный (регрессия на релевантность) — самый простой и самый
слабый. Попарный (RankNet) — учит порядок пар, что мы и посчитали. Списочный (LambdaMART,
ListNet) — оптимизирует метрику по всему списку сразу и обычно выигрывает.

**Что победило на практике.** LambdaMART — градиентный бустинг деревьев с лямбдами
LambdaRank — оставался лучшим методом на табличных признаках больше десяти лет и до сих пор
используется в проде как финальная ступень поверх нейронных признаков. Наша `alpha` — это
его вырожденный случай с одним признаком и линейной моделью.
</details>

---

## Задания — 15 мин

**Про самопроверку честно:** пройденная самопроверка не гарантирует, что задание сделано
осмысленно, но проваленная гарантирует, что где-то ошибка.

### Задание 1 · MaxSim своими руками

**Что сделать.** Реализуй `maxsim_pair(q_vecs, d_vecs)` — позднее взаимодействие для двух
наборов **нормированных** векторов: для каждого вектора запроса найти максимальный косинус
среди векторов документа и сложить. Проверь на игрушке лекции и **честно сравни** с наивной
альтернативой: усреднить векторы документа в один и посчитать один косинус.

**Что нужно получить.** `maxsim_pair` (функция), `mean_pool_score` — скор наивной альтернативы
на релевантном документе игрушки (`float`).

**Подсказка.** Матрица косинусов уже есть в `CB["docRel"]["sim"]`, считать эмбеддинги не нужно —
достаточно работать со строками этой матрицы.

**Прочитай до запуска.** Исходы:
* MaxSim релевантного выше, чем у нерелевантного, а наивное усреднение их путает — типичный
  случай, ради которого позднее взаимодействие и придумано;
* оба метода дают одинаковый порядок — на игрушке из трёх токенов это возможно, и вывод тогда
  «пример слишком мал, чтобы различить», а не «методы эквивалентны»;
* MaxSim ниже — почти наверняка ты берёшь максимум по столбцам, а не по строкам.

**Формулировка вывода.** Не «MaxSim лучше», а: **какое свойство запроса** делает разницу между
ними большой, и как это свойство проверить на своих данных.

In [ ]:
# --- твой код: ЗАДАНИЕ 1 ---
def maxsim_pair(sim_rows):
    ...

mean_pool_score = ...
# --- конец ---

assert abs(maxsim_pair(CB["docRel"]["sim"]) - CB["docRel"]["maxSim"]) < 1e-6, \
    "MaxSim релевантного не совпал с лекцией -- максимум берётся по СТРОКАМ"
assert abs(maxsim_pair(CB["docIrr"]["sim"]) - CB["docIrr"]["maxSim"]) < 1e-6, \
    "MaxSim нерелевантного не совпал с лекцией"
assert isinstance(mean_pool_score, float), "mean_pool_score -- число, скор наивной альтернативы"
n_q = len(CB["qTokens"])
assert abs(mean_pool_score - sum(sum(r) / len(r) for r in CB["docRel"]["sim"]) / 1) < 1e-6 \
    or abs(mean_pool_score * n_q - sum(sum(r) / len(r) for r in CB["docRel"]["sim"])) < 1e-6, \
    "усреднение считается по СТРОКАМ матрицы, затем суммируется или усредняется -- зафиксируй одно"
print(f"MaxSim: релевантный {maxsim_pair(CB['docRel']['sim']):.2f} · "
      f"нерелевантный {maxsim_pair(CB['docIrr']['sim']):.2f}")
print(f"наивное усреднение на релевантном: {mean_pool_score:.4f}")
RUN["task1"] = {"maxsim_rel": maxsim_pair(CB["docRel"]["sim"]), "mean_pool": mean_pool_score}

### Задание 2 · Цена подгонки на мелкой сетке

**Тезис.** *Чем мельче сетка перебора, тем больше оптимизм.* Проверим буквально: повторим
процедуру части 4 на сетке из 101 значения вместо 11 и сравним оптимизм.

**Что сделать.** Собери `FINE` — сетку `alpha` с шагом 0,01, подбери на половине-подборе,
померяй на отложенной половине и посчитай `optimism_fine` тем же способом, что в части 4.
**Честно сравни** с `optimism` на грубой сетке.

**Что нужно получить.** `optimism_fine` (`float`), `alpha_fine` (`float`).

**Подсказка.** `mrr_on(tune, a)` и `mrr_on(held, a)` уже написаны; менять надо только сетку.

**Прочитай до запуска.** Все исходы содержательны:
* `optimism_fine > optimism` — тезис подтверждён: больше попыток, больше везения;
* примерно равны — сетка уже плотнее, чем разрешение данных, и лишние точки ничего не меняют;
* `optimism_fine < optimism` — возможно при малой выборке случайно; это повод повторить
  на другом разбиении, а не вывод.

**Формулировка вывода.** Не «мелкая сетка вредна», а: **при каком отношении** числа
конфигураций к числу запросов подбор перестаёт быть измерением и становится подгонкой.

In [ ]:
# --- твой код: ЗАДАНИЕ 2 ---
FINE = ...
alpha_fine = ...
optimism_fine = ...
# --- конец ---

assert len(FINE) >= 100, "сетка должна быть мелкой -- не меньше сотни значений"
assert 0.0 <= alpha_fine <= 1.0, "alpha -- вес в [0,1]"
assert optimism_fine >= -1.0, "оптимизм -- разница двух MRR, он не может быть меньше -1"
print(f"грубая сетка ({len(ALPHAS)} точек): alpha {alpha_tuned}, оптимизм {optimism:+.4f}")
print(f"мелкая сетка ({len(FINE)} точек): alpha {alpha_fine}, оптимизм {optimism_fine:+.4f}")
print(f"разница оптимизмов: {optimism_fine - optimism:+.4f}")
RUN["task2"] = {"alpha_fine": alpha_fine, "optimism_fine": optimism_fine}

### Задание 3 · Словами: чем платит RRF

**Что сделать.** RRF складывает обратные ранги и не требует сопоставимых шкал — это его главное
удобство. Ответь **словами** на два вопроса:

1. Какую информацию RRF выбрасывает по сравнению со слиянием скоров, и в какой ситуации эта
   потеря дорого обходится? Приведи конкретный пример выдачи из двух систем, где RRF даст
   заведомо худший порядок, чем взвешенное слияние.
2. Мы получили, что RRF по полноте равен би-энкодеру и лучше BM25. Объясни, почему из этого
   **не следует**, что слияние полезно, и какой замер это бы показал.

**Прочитай до запуска.** `assert` проверяет объём, замену заглушки и что во втором пункте
названа конкретная альтернатива сравнения. Ответ без примера в первом пункте проверку пройдёт
и ревью не пройдёт.

**Формулировка вывода.** Не «RRF хуже», а: при каком свойстве скоров слияние величин
предпочтительнее слияния мест.

In [ ]:
# --- твой код: ЗАДАНИЕ 3 ---
ANSWER = """
Впиши ответ сюда: минимум 80 слов, оба пункта, в первом -- конкретный пример выдачи.
"""
# --- конец ---

assert len(ANSWER.split()) >= 80, "ответ короче 80 слов -- два пункта с примером так не уместить"
assert "Впиши ответ" not in ANSWER, "заглушка не заменена"
assert "би-энкодер" in ANSWER or "плотн" in ANSWER.lower(), \
    "второй пункт просит сравнить с ЛУЧШЕЙ из сливаемых систем -- назови её"
print(f"ответ принят: {len(ANSWER.split())} слов")

---

## Итог занятия — 5 мин

* Закрыли долг недели 4: дубликаты при слиянии реальны, и без дедупликации метрика выдаёт
  значения больше единицы, не падая.
* Посчитали **потолок объединения** — до всякой реализации. Он показывает, есть ли что брать,
  и стоит одну ячейку.
* Сверили с доской три способа быть «между»: MaxSim, разреженные обучаемые веса, RRF.
* Слили лексическое и плотное двумя путями и обнаружили, что оптимальный вес зависит
  от нормировки и от метрики — то есть «лучший вес» не свойство данных.
* Получили **первый за курс различимый** результат — и честно разобрали, что различимым
  оказалось не то, что мы хотели показать.
* Измерили оптимизм подгонки. Он сопоставим с самим эффектом.

**Ограничение нашего замера, которое надо назвать вслух.** Восемьдесят запросов, разделённых
пополам, дают сорок на каждую оценку — разрешение хуже, чем на неделе 7. Псевдозапросы
по-прежнему щедры к лексике. Скоры BM25 и косинусы приведены к общей шкале искусственно,
и выбор заполнителя для отсутствующих документов влияет на результат не меньше, чем `alpha`.

**Что мы будем и чего не будем замерять дальше.** На неделе 10 всё это ляжет на ANN-индекс,
и появится третья ось компромисса — память. Там же выяснится, что приближённый поиск сам
по себе теряет полноту, и потолок объединения придётся считать заново.

In [ ]:
best_orders = [fuse(i, alpha_tuned) for i in range(NQ)]
RUN["finished"] = True
(ARTIFACTS / "fusion.json").write_text(json.dumps({
    "alpha": alpha_tuned, "norm": "z", "k": K, "rrf_k": RRF_K,
    "tune_queries": tune, "held_queries": held,
    "fused_top100": best_orders,
    "rrf_top100": RRF_RUN,
    "run": RUN,
}, ensure_ascii=False), encoding="utf-8")
(ARTIFACTS / "run-fusion.json").write_text(
    json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"слияние -> {ARTIFACTS / 'fusion.json'} "
      f"(alpha={alpha_tuned}, разбиение запросов внутри)")
print(f"замеры  -> {ARTIFACTS / 'run-fusion.json'} ({len(RUN)} ключей)")
print("на неделе 10 lab-ann построит индекс под ЭТО ранжирование")

**Что видно.** В артефакт легли не только выдачи, но и **разбиение запросов** на подбор
и проверку. Сравнивать надо не размеры файлов, а **что из этого нельзя восстановить**: выдачи
пересчитываются, а вот какое разбиение было использовано при подборе — нет, и без него любое
будущее сравнение станет нечестным. Механизм прямой: если на неделе 10 кто-то померяет качество
на запросах, по которым сегодня подбиралась `alpha`, он получит тот самый оптимизм, который
мы только что измерили. Чего этот вывод НЕ показывает: что `alpha` подойдёт другому корпусу —
она подобрана здесь и здесь же проверена. Что делать: на неделе 10 мерить **только**
на отложенной половине, взяв её из этого файла.

<details><summary>Что понадобится на неделе 10 — и почему разбиение запросов важнее выдач</summary>

В `fusion.json` легли выдачи, вес и разбиение запросов. Последнее — самое ценное, хотя весит
меньше всего.

**Почему разбиение.** `alpha = 0,2` подобрана на конкретных сорока запросах. Любой замер
на этих же запросах на неделе 10 унаследует оптимизм, который мы сегодня измерили, и будет
завышен примерно на две сотых. Восстановить разбиение постфактум невозможно: оно порождено
перемешиванием с сидом, и стоит кому-то поменять сид или число запросов — соответствие
потеряно навсегда.

**Что делает неделя 10.** Строит ANN-индекс поверх тех же эмбеддингов и меряет, сколько
полноты теряется от приближённого поиска. Ключевой вопрос — компромисс «память против полноты»,
и мерить его надо на **отложенной** половине, иначе потеря полноты смешается с оптимизмом
подбора и будет выглядеть меньше, чем есть.

**Тонкость, которая всплывёт.** ANN-индекс возвращает приближённый список кандидатов, то есть
меняет вход слияния. Значит, `alpha`, подобранная на точном поиске, формально уже не оптимальна.
Правильно было бы подобрать её заново поверх ANN — и это ещё одна конфигурация в переборе,
и ещё немного оптимизма. Практический компромисс: проверить, что качество при старой `alpha`
не упало, и не трогать её, пока не упало.

**Общий принцип, который стоит унести.** Разбиение данных — часть артефакта, а не деталь
прогона. Как только ты что-то подобрал, набор, на котором подбирал, становится «сожжённым»
для всех последующих замеров, и знать, какой именно набор сожжён, важнее, чем знать результат.
</details>

<details><summary>Ограничения этого семинара, которые надо назвать вслух</summary>

Полный список того, где мы срезали угол, и в какую сторону это смещает выводы.

**Восемьдесят запросов, разделённых пополам.** Сорок на подбор, сорок на проверку. Разрешение
хуже, чем на неделе 7, где было восемьдесят. Смещение: толкает к выводу «не отличимо», и три
из четырёх сравнений действительно не отличимы.

**Псевдозапросы.** Те же, что на неделе 7, и то же смещение — лексика в выигрыше. Значит,
оптимальная `alpha` у нас завышена относительно того, что было бы на настоящих запросах:
BM25 здесь сильнее, чем заслуживает.

**Один заполнитель для отсутствующих.** Мы поставили `−4` для z-нормировки и ноль для min-max
и не проверили, насколько результат от этого зависит. Это полноценный второй параметр,
который мы не подбирали и не обсуждали количественно.

**Обрезка объединения по глубине сто.** Два списка по сто не помещаются в сто позиций, и часть
потолка недостижима по построению. Мы это назвали, но не измерили: сколько именно теряется
на обрезке, осталось неизвестным.

**Слияние только двух систем.** Кросс-энкодер недели 7 в слиянии не участвовал вовсе — его
выдача вчетверо короче, и включить её честно означало бы отдельную работу с разными глубинами.
Это сознательный отказ.

**Все игрушки — из лекции.** ColBERT, SPLADE и LTR мы проверили на трёхтокенных примерах.
Они доказывают, что мы правильно поняли формулу, и ничего не говорят о поведении методов
на данных.

<summary>Как сделать правильно, если есть бюджет</summary>
Пятикратная кросс-валидация вместо разделения пополам, прогон по заполнителю вместе с прогоном по
`alpha`, объединение и замер на глубине двести, и третья система в слиянии. Это часы счёта,
и почти всё — механическая работа поверх уже написанного.
</details>

---

## Решения

**Подглядеть — не поражение. Поражение — уйти с занятия, не поняв, где был затык.**

<details><summary>Задание 1 · MaxSim</summary>

```python
def maxsim_pair(sim_rows):
    return sum(max(row) for row in sim_rows)

mean_pool_score = sum(sum(r) / len(r) for r in CB["docRel"]["sim"])
```

Свойство запроса, делающее разницу большой: **многоаспектность**. Если запрос состоит
из нескольких независимых требований («река», «наводнение»), усреднение размажет их в один
вектор, и документ, отвечающий на одно требование очень сильно, обгонит документ, отвечающий
на все понемногу. MaxSim этого не допускает: каждый токен запроса ищет себе пару отдельно.

Проверить на своих данных можно так: взять запросы длиннее пяти слов, посчитать разброс
косинусов между токенами запроса и центроидом документа. Большой разброс означает, что запрос
многоаспектен, и позднее взаимодействие окупится. Малый — усреднение ничего не теряет,
и платить памятью незачем.
</details>

<details><summary>Задание 2 · цена мелкой сетки</summary>

```python
FINE = [round(i / 100, 2) for i in range(101)]
alpha_fine = max(FINE, key=lambda a: mrr_on(tune, a))
optimism_fine = mrr_on(tune, alpha_fine) - mrr_on(held, alpha_fine)
```

Отношение, при котором подбор превращается в подгонку, задаётся не абсолютным числом
конфигураций, а их числом относительно **разрешения** данных. Если соседние значения `alpha`
дают разницу меньше, чем половина ширины доверительного интервала, то различить их
невозможно в принципе, и любой перебор внутри такого шага выбирает шум.

Практический ориентир из части 4: ширина интервала у нас порядка нескольких сотых, значит
осмысленный шаг `alpha` — не мельче 0,1, то есть наша грубая сетка уже на пределе. Сто одна
точка — это сто одна попытка угадать на сорока запросах.
</details>

<details><summary>Задание 3 · чем платит RRF</summary>

**Первый пункт.** RRF выбрасывает **величину** уверенности. Пример: система A вернула документ
X с огромным отрывом от второго места, система B поставила X на десятое место с почти таким же
скором, как у соседей. По скорам X должен победить — его нашли уверенно. По рангам X получает
`1/61 + 1/70`, что меньше, чем у документа, стоящего третьим и четвёртым у обеих систем без
всякой уверенности. RRF предпочтёт согласованную посредственность уверенной находке.

**Второй пункт.** Из «RRF лучше BM25» не следует пользы слияния, потому что тот же результат
достигается выбрасыванием BM25 и использованием одного би-энкодера. Показал бы это замер,
который мы и сделали: сравнение гибрида с **лучшей** из сливаемых систем, а не с худшей.
Оно дало ровно ноль по полноте.

**Когда слияние величин предпочтительнее.** Когда скоры систем **калиброваны** — то есть
их величина сопоставима между запросами, а не только внутри одного. Тогда уверенность несёт
информацию, и выбрасывать её жалко. Если калибровки нет (а у BM25 её нет: скор зависит
от длины запроса), RRF безопаснее именно потому, что не полагается на то, чего нет.
</details>

---

## Литература

* **Cormack, Clarke & Buettcher (2009), «Reciprocal Rank Fusion Outperforms Condorcet and
  Individual Rank Learning Methods»** — откуда взялись RRF и константа 60. Короткая и
  показательная: метод в одну строку обыгрывает обученные комбинации.
* **Khattab & Zaharia (2020), «ColBERT»** и **Santhanam et al. (2022), «ColBERTv2»** — позднее
  взаимодействие и то, как его довели до приемлемого размера индекса.
* **Formal, Piwowarski & Clinchant (2021), «SPLADE»** — обучаемые разреженные представления
  и регуляризация, удерживающая разреженность.
* **Burges (2010), «From RankNet to LambdaRank to LambdaMART»** — вывод того, что мы считали
  на игрушке, из первых принципов. Лучший текст про то, почему метрика входит в градиент.
* **Bruch et al. (2023), «An Analysis of Fusion Functions for Hybrid Retrieval»** — аккуратное
  сравнение RRF и слияния скоров, включая вопрос нормировки. Прямо про часть 3.
* **Лекция L12 «Альянс»** и `data/l8-*.json` — числа, с которыми мы сверялись.

**Дальше по курсу.** L13 объясняет, как искать среди миллионов векторов, не перебирая их все,
и неделя 10 добавит к нашим двум осям компромисса третью — память.

## Дамп прогона

Правило 10.5: занятие не считается прогнанным, пока его числа не лежат в файле рядом
с конфигурацией рантайма. Ячейка ниже собирает все численные результаты ноутбука —
от сида до финальных метрик — и кладёт их в `runs/hw-alliance.json`. Это и есть
доказательство прогона: разбор сверяется с файлом, а не с памятью автора.

In [ ]:
# Дамп прогона — все числовые результаты + конфигурация рантайма (правило 10.5).
import json as _json, os as _os, sys as _sys, platform as _pl, pathlib as _pathlib

_runtime = {"python": _sys.version.split()[0], "platform": _pl.platform()}
_torch = _sys.modules.get("torch")   # НЕ импортируем сами: рамка 7.4 — сид и пин
if _torch is not None:                # обязателен только там, где ноутбук torch ИСПОЛЬЗУЕТ
    _runtime["torch"] = _torch.__version__
    _runtime["gpu"] = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else None
else:
    import subprocess as _sp
    try:
        _q = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=5)
        _runtime["gpu"] = (_q.stdout.strip().splitlines() or [None])[0] if _q.returncode == 0 else None
    except Exception:
        _runtime["gpu"] = None

def _plain(v):
    try:
        import numpy as _np
        if isinstance(v, _np.integer): return int(v)
        if isinstance(v, _np.floating): return float(v)
    except Exception:
        pass
    return v

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

_metrics = {}
for _k, _v in sorted(globals().items()):
    if _k.startswith("_") or (len(_k) == 1 and _k.islower()):
        continue                       # служебные имена и счётчики циклов
    _v = _plain(_v)
    if _num(_v):
        _metrics[_k] = _v
    elif isinstance(_v, dict) and 0 < len(_v) <= 64 and all(_num(_plain(_x)) for _x in _v.values()):
        _metrics[_k] = {str(_kk): _plain(_vv) for _kk, _vv in _v.items()}
    elif isinstance(_v, (list, tuple)) and 0 < len(_v) <= 64 and all(_num(_plain(_x)) for _x in _v):
        _metrics[_k] = [_plain(_x) for _x in _v]

_out = _pathlib.Path(_os.environ.get("RUNS_DIR", "runs")); _out.mkdir(parents=True, exist_ok=True)
_path = _out / "hw-alliance.json"
_json.dump({"notebook": "hw-alliance", "runtime": _runtime, "metrics": _metrics},
           open(_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1, sort_keys=True)
print(f"дамп: {_path} · величин: {len(_metrics)} · рантайм: {_runtime['gpu'] or 'CPU'}")

**Что видно.** В дампе — конфигурация прогона и все скалярные результаты по именам
переменных. Сравнивать надо не тайминги — они свойство рантайма, и на T4, A100 и CPU
законно разные, — а метрики качества: при одном сиде они обязаны совпасть до знака.
Если твой прогон разошёлся с эталонным в качестве, а не во времени, — это находка,
неси её на занятие.